# CS781 Phase 6 — Abstractive Summarization

**Task:** generate one concise abstractive summary for each (long) legal document.
There is no public dev split: a deterministic 90/10 train/validation split is created from `train.json`.

**Formulations**
- *Causal* (decoder / SSM / CNN): one sequence `[BOS] + document (first MAX_SOURCE_TOKENS tokens) + " Summary:" + summary + [EOS]`; loss only on summary tokens; greedy/beam generation from the prompt with the Phase-1 LM head.
- *Encoder-decoder* (`encdec`): document → encoder, summary → decoder (teacher forcing). `STANDARD_ENCDEC = True` uses the plain token-level encoder;
  `False` uses the hierarchical variant (every 1000-token chunk encoded separately, `memory_slots` token states per chunk + a learned chunk-position embedding).

**Output (`prediction.json`):** a JSON **list** with exactly one generated summary per test document:

```json
[{"id": "doc_001", "summary": "Generated text..."}]
```

**Primary metric:** BERTScore F1 (secondary: ROUGE-1 F1, ROUGE-L F1). BERTScore/ROUGE are used only to *evaluate* generated text;
the summarizer is built only from the Phase-1 models and tokenizer.

**Adaptation techniques**

| # | Key | Technique | Implementation |
|---|-----|-----------|----------------|
| 1 | `lora`  | LoRA — additive low-rank update `W₀x + s·BAx` | `peft` — `LoraConfig(use_rslora=False)` |
| 2 | `lorma` | **LORMA (compulsory)** — low-rank **multiplicative** update `(I + s·BA)·W₀x` | Custom `LowRankMulLinear` injection |
| 3 | `kd`    | **Knowledge Distillation (compulsory)** | `DistillationExperiment` + token-level `sequence_kd_loss` (section 15) |

**How to run.** Set paths, the formulation switches and run flags in section 2, then *Run All*. The default flags reproduce the best
submission: preflight checks → train `exp_cfg` (decoder + LORMA) → greedy generation for the test set → `submission.zip`.
Knowledge distillation runs only when `RUN_KD = True`.

| Stage | Section |
|---|---|
| Environment and imports | 1 |
| Paths and configuration | 2 |
| Data loading, split and validation | 3 |
| Tokenizer | 4 |
| Dataset and DataLoaders | 5 |
| Phase-1 backbone definitions + decoding | 6 |
| Backbone registry and checkpoint checks | 7 |
| Phase-6 model wrapper | 8 |
| Long-document handling and length budget | 9 |
| Adaptation methods | 10 |
| Experiment configuration | 11 |
| Training, evaluation and submission utilities | 12 |
| Preflight tests (optional, lightweight) | 13 |
| Main training experiment | 14 |
| Knowledge distillation (optional) | 15 |
| Test inference (generation) | 16 |
| Submission validation and ZIP | 17 |

## 1. Environment and imports

In [ ]:
!pip install -q huggingface_hub safetensors peft rouge_score bert_score

In [ ]:
import os, json, random, math, time, zipfile, copy, gc
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, field, replace
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

# Built-in PEFT support (LoRA, rsLoRA, IA3)
import peft
from peft import LoraConfig, IA3Config, get_peft_model

# Evaluation-only libraries (Phase-6 metrics)
from rouge_score import rouge_scorer
from bert_score import score as bert_score

print("torch:", torch.__version__)
print("peft :", peft.__version__)

# ------------------------------------------------------------
# Determinism
# ------------------------------------------------------------
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 2. Paths and configuration

All Kaggle paths, the summarization formulation and the run flags live here.

In [ ]:
# ------------------------------------------------------------
# Phase-1 tokenizer and backbone checkpoints
# ------------------------------------------------------------
TOKENIZER_PATH     = "/kaggle/input/datasets/richierich6/tokenizer6/legal_bpe.json"
DECODER_CHECKPOINT = "/kaggle/input/datasets/richierich6/deconly6/pytorch_model.bin"
SSM_CHECKPOINT     = "/kaggle/input/datasets/sg123321/ssmonly/model.safetensors"
ENCDEC_CHECKPOINT  = "/kaggle/input/datasets/richierich6/encdec"                  # directory or file
CNN_CHECKPOINT     = "/kaggle/input/datasets/sg123321/cnn-atten/model.safetensors"

# Trained Phase-6 teacher for knowledge distillation: decoder + LORMA (see teacher_cfg, section 15).
KD_TEACHER_CHECKPOINT = "/kaggle/input/datasets/richierich6/dec-lorma/dec_lorma.pt"

# Checkpoint used by test inference when the model was not trained in this session.
# None -> <run_dir>/best_model.pt of the selected configuration.
INFERENCE_CHECKPOINT = None

# ------------------------------------------------------------
# Formulation (must match exp_cfg.backbone_name; checked by check_length_budget)
# ------------------------------------------------------------
ENCODER_DECODER = False   # True only when exp_cfg.backbone_name == "encdec"
STANDARD_ENCDEC = True    # encdec: True = token-level encoder (used by the recorded encdec runs),
                          #         False = hierarchical chunk memory

# ------------------------------------------------------------
# Lengths / loader settings
# ------------------------------------------------------------
MAX_SOURCE_TOKENS = 1000          # document tokens in the prompt (causal) / per encoder input (encdec)
MAX_TARGET_TOKENS = 1000          # summary tokens used for training
CHUNK_OVERLAP = 128               # hierarchical encdec only
CHUNK_STRIDE = MAX_SOURCE_TOKENS - CHUNK_OVERLAP
SUMMARY_PROMPT = " Summary:"
BATCH_SIZE = 1
NUM_WORKERS = 0                   # 0 is the safe default on Kaggle

# ------------------------------------------------------------
# What a "Run All" executes (defaults reproduce the best submission: decoder + LORMA)
# ------------------------------------------------------------
RUN_PREFLIGHT = True                                      # section 13 (lightweight)
PREFLIGHT_BACKBONES = ["decoder", "encdec", "cnn", "ssm"]  # backbones checked in section 13
RUN_TINY_OVERFIT = False                                  # section 13
RUN_TRAINING = True                                       # section 14: trains exp_cfg
RUN_KD = False                                            # section 15
RUN_INFERENCE = True                                      # sections 16-17
INFERENCE_SOURCE = "main"                                 # "main" = exp_cfg model, "kd" = KD student

In [ ]:
# ============================================================
# HUGGING FACE SETUP
# ============================================================
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download, list_repo_files

HF_TOKEN = UserSecretsClient().get_secret("hf_token_rp")
login(token=HF_TOKEN, add_to_git_credential=False)

# ============================================================
# PHASE 6 DATASET FROM HUGGING FACE
# ============================================================
DATA_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE_DIR = "Phase_6_Summ"

REPO_FILES = list_repo_files(DATA_REPO, repo_type="dataset", token=HF_TOKEN)
PHASE_FILES = [f for f in REPO_FILES if f.startswith(PHASE_DIR + "/")]
print(f"{PHASE_DIR} files in repo:")
for f in PHASE_FILES:
    print("  ", f)


def get_data_file(filename):
    """Download `filename`; if that exact path is absent, fall back to a UNIQUE basename match inside PHASE_DIR."""
    if filename not in REPO_FILES:
        base = filename.split("/")[-1]
        matches = [f for f in PHASE_FILES if f.split("/")[-1] == base]
        if len(matches) != 1:
            raise FileNotFoundError(f"'{filename}' not in repo; basename matches in {PHASE_DIR}: {matches}")
        print(f"[note] '{filename}' not found, using '{matches[0]}'")
        filename = matches[0]
    return Path(
        hf_hub_download(
            repo_id=DATA_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    )


TRAIN_FILE = get_data_file(
    "Phase_6_Summ/Train/train.json"
)

TEST_FILE = get_data_file(
    "Phase_6_Summ/Test/test.json"
)

for name, p in [
    ("TRAIN_FILE", TRAIN_FILE),
    ("TEST_FILE", TEST_FILE),
]:
    print(f"{name:16s}: {p}")

## 3. Data loading, split and validation

Inspect the raw JSON first. Each training item has an id, the source `document` and the reference `summary`; test items have id + document.
**No public dev split** → `deterministic_split()` holds out 10% of training documents (seed 42, split by document id, so no document appears in both).

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def describe_json(obj, name, depth=0, max_items=2):
    """Print the nested structure of a JSON object (types, sizes, first keys) to discover the schema."""
    pad = "  " * depth
    if isinstance(obj, dict):
        print(f"{pad}{name}: dict[{len(obj)}] keys={list(obj.keys())[:8]}")
        for k in list(obj.keys())[:max_items]:
            describe_json(obj[k], repr(k), depth + 1, max_items)
    elif isinstance(obj, list):
        print(f"{pad}{name}: list[{len(obj)}]")
        for i, x in enumerate(obj[:max_items]):
            describe_json(x, f"[{i}]", depth + 1, max_items)
    else:
        s = repr(obj)
        print(f"{pad}{name}: {s[:120]}{'...' if len(s) > 120 else ''}")


train_raw = load_json(TRAIN_FILE)
test_raw  = load_json(TEST_FILE)

describe_json(train_raw, "train")
describe_json(test_raw, "test")

In [ ]:
# Convert each raw split into a uniform list of documents:
#     {"id": str, "document": str, "summary": str}     ("summary" absent for test)
# - field names: use whatever the inspection above shows (e.g. "document"/"text",
#   "summary"/"abstract") and map them to these keys
# - keep ids EXACTLY as given — they are echoed in the submission
# - assert ids are unique; report empty documents / summaries
# ------------------------------------------------------------


def normalize_split(raw, has_summary=True):
    docs, seen = [], set()
    empty_doc = empty_sum = 0

    for x in raw:
        id_ = x["id"]
        assert isinstance(id_, str), f"ID must be string: {id_}"
        assert id_ not in seen, f"Duplicate ID: {id_}"
        seen.add(id_)

        doc = x["document"]
        assert isinstance(doc, str), f"Invalid document: {id_}"
        empty_doc += not bool(doc.strip())

        item = {"id": id_, "document": doc}

        if has_summary:
            summ = x["summary"]
            assert isinstance(summ, str), f"Invalid summary: {id_}"
            empty_sum += not bool(summ.strip())
            item["summary"] = summ

        docs.append(item)

    print(f"Total: {len(docs)}, Empty documents: {empty_doc}, Empty summaries: {empty_sum}")
    return docs



def deterministic_split(items, val_fraction=0.10, seed=SEED):
    """Deterministic train/val split by id: sort ids, shuffle with a fixed seed, hold out `val_fraction`."""
    ids = sorted(x["id"] for x in items)
    assert len(ids) == len(set(ids)), "duplicate ids"
    rng = random.Random(seed)
    rng.shuffle(ids)
    n_val = max(1, int(round(len(ids) * val_fraction)))
    val_ids = set(ids[:n_val])
    train_part = [x for x in items if x["id"] not in val_ids]
    val_part = [x for x in items if x["id"] in val_ids]
    return train_part, val_part


all_train_docs = normalize_split(train_raw)
test_docs = normalize_split(test_raw, has_summary=False)

train_docs, val_docs = deterministic_split(all_train_docs, val_fraction=0.10, seed=SEED)
assert not ({d["id"] for d in train_docs} & {d["id"] for d in val_docs}), "train/val leakage"

print("Train docs:", len(train_docs))
print("Val docs  :", len(val_docs))
print("Test docs :", len(test_docs))

In [ ]:
# ------------------------------------------------------------
# Data validation (raises on any inconsistency)
# ------------------------------------------------------------
assert [d["id"] for d in test_docs] == [x["id"] for x in test_raw], "test IDs lost or reordered"
assert all("summary" not in d for d in test_docs), "test documents must not carry summaries"
assert len(train_docs) + len(val_docs) == len(all_train_docs)
assert {d["id"] for d in train_docs}.isdisjoint({d["id"] for d in val_docs}), "train/val leakage"
assert {d["id"] for d in test_docs}.isdisjoint({d["id"] for d in all_train_docs}), "test/train ID overlap"
_split_again = deterministic_split(all_train_docs, val_fraction=0.10, seed=SEED)
assert [d["id"] for d in _split_again[1]] == [d["id"] for d in val_docs], "split is not deterministic"
print("Data validation passed.")

In [ ]:
def length_stats(docs, key, n=200):
    """Approximate token lengths on a sample."""
    lens = [len(tokenizer.encode(d[key])) for d in docs[:n]]
    return {"sampled": len(lens), "mean": float(np.mean(lens)), "p50": float(np.median(lens)),
            "p90": float(np.percentile(lens, 90)), "max": int(max(lens))}

## 4. Tokenizer

Same `ByteLevelBPE` class and `legal_bpe.json` as Phase 3 (unchanged).

In [ ]:
#BPE tokenizer

def train_tokenizer(train_text, vocab_size):
    tokenizer = ByteLevelBPE(vocab_size=vocab_size)
    tokenizer.train([train_text])
    return tokenizer


import json
from collections import Counter, defaultdict
from tqdm.auto import tqdm


BPE_SPECIALS = [
    "<pad>",
    "<unk>",
    "<s>",
    "</s>"
]

# End-of-word marker.
# UTF-8 bytes are integers 0..255, so 256 cannot collide.
EOW = 256


class ByteLevelBPE:
    """
    Deterministic byte-level BPE tokenizer.

    - Initial units are UTF-8 bytes.
    - EOW marks the end of each word.
    - Case is preserved.
    - BPE never crosses word boundaries.
    - Ties are broken lexicographically.
    - Four special tokens are fixed at the beginning.
    """

    def __init__(self, vocab_size=5000):

        self.vocab_size = vocab_size
        self.merge_ranks = {}
        self.specials = list(BPE_SPECIALS)

        # List of learned merges.
        #
        # Each element is:
        #     (left_symbol, right_symbol)
        #
        self.merges = []

        # Vocabulary.
        self.itos = []
        self.stoi = {}

        # Internal merge rules.
        #
        # (left, right) -> merged_symbol
        #
        self.merge_rules = {}

        # Final representations of words seen during training.
        self.word_cache = {}
        self.token_to_symbol = {}

    # --------------------------------------------------------
    # Convert a byte/symbol tuple into a deterministic string
    # --------------------------------------------------------
    @staticmethod
    def string_to_symbol(token):
        # Standalone end-of-word marker
        if token == "</w>":
            return (EOW,)
    
        # Merged symbol ending with EOW
        if token.endswith("</w>>"):
            hex_part = token[3:-5]
            byte_part = tuple(
                int(hex_part[i:i+2], 16)
                for i in range(0, len(hex_part), 2)
            )
            return byte_part + (EOW,)
    
        # Ordinary byte or merged byte symbol
        hex_part = token[3:-1]
        return tuple(
            int(hex_part[i:i+2], 16)
            for i in range(0, len(hex_part), 2)
        )
    @staticmethod
    def symbol_to_string(symbol):

        # EOW is always the last symbol when present.
        if symbol and symbol[-1] == EOW:

            byte_part = symbol[:-1]

            if len(byte_part) == 0:
                return "</w>"

            hex_part = "".join(
                f"{b:02x}"
                for b in byte_part
            )

            return f"<0x{hex_part}</w>>"

        hex_part = "".join(
            f"{b:02x}"
            for b in symbol
        )

        return f"<0x{hex_part}>"
    def _build_merge_ranks(self):
        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(self.merge_rules)
        }
    # --------------------------------------------------------
    # Deterministic ordering key for symbols
    # --------------------------------------------------------

    @staticmethod
    def symbol_key(symbol):
        return tuple(symbol)

    # --------------------------------------------------------
    # Convert a word into initial UTF-8 byte symbols
    # --------------------------------------------------------

    @staticmethod
    def word_to_symbols(word):

        byte_values = list(
            word.encode("utf-8")
        )

        symbols = [
            (b,)
            for b in byte_values
        ]

        # Explicit word boundary.
        symbols.append((EOW,))

        return symbols

    # --------------------------------------------------------
    # Count all pairs once initially
    # --------------------------------------------------------

    def _initial_pair_counts(
        self,
        word_symbols,
        word_counts
    ):

        pair_counts = Counter()

        # pair_words tells us which words contain a pair.
        pair_words = defaultdict(set)

        for word, symbols in word_symbols.items():

            freq = word_counts[word]

            for i in range(len(symbols) - 1):

                pair = (
                    symbols[i],
                    symbols[i + 1]
                )

                pair_counts[pair] += freq
                pair_words[pair].add(word)

        return pair_counts, pair_words

    # --------------------------------------------------------
    # Merge one pair in one word
    # --------------------------------------------------------

    @staticmethod
    def _merge_word(symbols, pair):

        left, right = pair

        new_symbol = left + right

        result = []

        i = 0

        while i < len(symbols):

            if (
                i + 1 < len(symbols)
                and symbols[i] == left
                and symbols[i + 1] == right
            ):

                result.append(new_symbol)

                i += 2

            else:

                result.append(symbols[i])

                i += 1

        return result

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    def train(self, texts):

        print("=" * 70)
        print("Training BYTE-LEVEL BPE")
        print("=" * 70)

        # ----------------------------------------------------
        # Count unique words.
        # ----------------------------------------------------

        word_counts = Counter()

        for text in texts:
            if not isinstance(text, str):
                continue

            for word in text.split():

                word_counts[word] += 1

        print(
            f"Unique words: {len(word_counts):,}"
        )

        # ----------------------------------------------------
        # Initial representation of each word.
        # ----------------------------------------------------

        word_symbols = {
            word: self.word_to_symbols(word)
            for word in word_counts
        }

        # ----------------------------------------------------
        # Initial vocabulary symbols.
        # ----------------------------------------------------

        initial_symbols = [
            (b,)
            for b in range(256)
        ]

        initial_symbols.append((EOW,))

        # Four specials + initial byte symbols.
        current_vocab_size = (
            len(BPE_SPECIALS)
            + len(initial_symbols)
        )

        print(
            f"Initial vocabulary size: "
            f"{current_vocab_size:,}"
        )

        if current_vocab_size > self.vocab_size:

            raise ValueError(
                "Initial byte vocabulary already exceeds "
                "the requested vocabulary size."
            )

        # ----------------------------------------------------
        # Count pairs once.
        # ----------------------------------------------------

        pair_counts, pair_words = \
            self._initial_pair_counts(
                word_symbols,
                word_counts
            )

        # ----------------------------------------------------
        # Main BPE loop.
        # ----------------------------------------------------

        merge_no = 0

        while current_vocab_size < self.vocab_size:

            if not pair_counts:
                break

            # Ignore pairs occurring only once.
            valid_pairs = [
                pair
                for pair, count in pair_counts.items()
                if count > 1
            ]

            if not valid_pairs:
                print(
                    "No pair occurs more than once. "
                    "Stopping early."
                )
                break

            # Highest frequency first.
            # Lexicographically smallest pair on ties.
            best_pair = min(
                valid_pairs,
                key=lambda pair: (
                    -pair_counts[pair],
                    self.symbol_key(pair[0]),
                    self.symbol_key(pair[1])
                )
            )

            best_count = pair_counts[best_pair]

            left, right = best_pair
            new_symbol = left + right

            # ------------------------------------------------
            # Record merge.
            # ------------------------------------------------

            self.merge_rules[best_pair] = new_symbol

            self.merges.append(
                (
                    self.symbol_to_string(left),
                    self.symbol_to_string(right)
                )
            )

            # ------------------------------------------------
            # Words affected by this merge.
            # ------------------------------------------------

            affected_words = list(
                pair_words.get(best_pair, set())
            )

            # Remove the selected pair's old contribution.
            del pair_counts[best_pair]

            pair_words.pop(
                best_pair,
                None
            )

            # ------------------------------------------------
            # Update each affected word.
            # ------------------------------------------------

            for word in affected_words:

                old_symbols = word_symbols[word]

                old_pairs = []

                for i in range(
                    len(old_symbols) - 1
                ):

                    old_pairs.append(
                        (
                            old_symbols[i],
                            old_symbols[i + 1]
                        )
                    )

                # Remove old pair contributions.
                freq = word_counts[word]

                for pair in old_pairs:

                    pair_counts[pair] -= freq

                    if pair_counts[pair] <= 0:
                        pair_counts.pop(
                            pair,
                            None
                        )

                    if pair in pair_words:
                        pair_words[pair].discard(word)

                        if not pair_words[pair]:
                            pair_words.pop(
                                pair,
                                None
                            )

                # Perform merge.
                new_symbols = self._merge_word(
                    old_symbols,
                    best_pair
                )

                word_symbols[word] = new_symbols

                # Add new pair contributions.
                new_pairs = []

                for i in range(
                    len(new_symbols) - 1
                ):

                    new_pairs.append(
                        (
                            new_symbols[i],
                            new_symbols[i + 1]
                        )
                    )

                for pair in new_pairs:

                    pair_counts[pair] += freq
                    pair_words[pair].add(word)

            current_vocab_size += 1
            merge_no += 1

            if merge_no % 250 == 0:

                print(
                    f"Merges: {merge_no:4d} | "
                    f"Vocab: {current_vocab_size:5d} | "
                    f"Best pair frequency: {best_count:,}"
                )

        # ----------------------------------------------------
        # Construct final vocabulary.
        #
        # First:
        #   <pad>, <unk>, <s>, </s>
        #
        # Then initial symbols.
        #
        # Then symbols introduced by merges in merge order.
        # ----------------------------------------------------

        merged_symbols = list(
            self.merge_rules.values()
        )

        self.itos = (
            list(BPE_SPECIALS)
            +
            [
                self.symbol_to_string(s)
                for s in initial_symbols
            ]
            +
            [
                self.symbol_to_string(s)
                for s in merged_symbols
            ]
        )

        self.token_to_symbol = {}

        for symbol in initial_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        for symbol in merged_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        # Remove duplicates while preserving order.
        final_itos = []
        seen = set()

        for token in self.itos:

            if token not in seen:

                seen.add(token)
                final_itos.append(token)

        self.itos = final_itos[
            :self.vocab_size
        ]

        self.stoi = {
            token: i
            for i, token in enumerate(self.itos)
        }

        # Save representations of training words.
        self.word_cache = word_symbols

        print(
            "\nTraining complete."
        )

        print(
            f"Final vocabulary size: "
            f"{len(self.itos):,}"
        )

        print(
            f"Number of merges: "
            f"{len(self.merges):,}"
        )
        self._build_merge_ranks()
        return self

    # --------------------------------------------------------
    # Tokenize a word
    # --------------------------------------------------------

    def tokenize_word(self, word):
        # Use cached representation for training words.
        if word in self.word_cache:
            symbols = self.word_cache[word]
        else:
            symbols = self.word_to_symbols(word)
    
            while len(symbols) > 1:
                # Find adjacent pairs that have learned merges.
                candidates = [
                    (self.merge_ranks[(symbols[i], symbols[i + 1])], i)
                    for i in range(len(symbols) - 1)
                    if (symbols[i], symbols[i + 1]) in self.merge_ranks
                ]
    
                if not candidates:
                    break
    
                # Apply the earliest learned merge.
                _, best_idx = min(candidates)
    
                pair = (symbols[best_idx], symbols[best_idx + 1])
                new_symbol = self.merge_rules[pair]
    
                # Merge all non-overlapping occurrences.
                symbols = self._merge_word(symbols, pair)
    
        return [
            self.symbol_to_string(symbol)
            for symbol in symbols
        ]

    # --------------------------------------------------------
    # Tokenize complete sentence
    # --------------------------------------------------------

    def tokenize(self, text):

        tokens = []

        for word in text.split():

            tokens.extend(
                self.tokenize_word(word)
            )

        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)

        ids = []
        for token in tokens:
            if token in self.stoi:
                ids.append(self.stoi[token])
            else:
                # unknown token
                ids.append(self.stoi["<unk>"])

        return ids

    def decode(self, ids):
      byte_values = []

      for idx in ids:
          token = self.itos[idx]

          if token in BPE_SPECIALS:
              continue

          symbol = self.token_to_symbol[token]

          for b in symbol:
              if b == EOW:
                  byte_values.append(ord(" "))
              else:
                  byte_values.append(b)

      # Remove the final space
      if byte_values and byte_values[-1] == ord(" "):
          byte_values.pop()

      return bytes(byte_values).decode("utf-8")

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = data["merges"]
        tokenizer.itos = data["itos"]
        tokenizer.stoi = {
            token: i for i, token in enumerate(tokenizer.itos)
        }
    
        # Initial symbols
        initial_symbols = [(b,) for b in range(256)]
        initial_symbols.append((EOW,))
    
        tokenizer.token_to_symbol = {
            tokenizer.symbol_to_string(symbol): symbol
            for symbol in initial_symbols
        }
    
        # Reconstruct merge rules in training order
        for left_str, right_str in tokenizer.merges:
            left = tokenizer.string_to_symbol(left_str)
            right = tokenizer.string_to_symbol(right_str)
            merged = left + right
    
            tokenizer.merge_rules[(left, right)] = merged
            merged_str = tokenizer.symbol_to_string(merged)
            tokenizer.token_to_symbol[merged_str] = merged
        tokenizer._build_merge_ranks()  
        return tokenizer

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    def save(self, path):

        data = {
            "merges": self.merges,
            "itos": self.itos,
            "vocab_size": self.vocab_size
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False
            )

        print(
            f"Saved: {path}"
        )

In [ ]:
tokenizer = ByteLevelBPE.load(TOKENIZER_PATH)

print("Vocabulary size:", len(tokenizer.itos))
print("Number of merges:", len(tokenizer.merges))

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

print("PAD/UNK/BOS/EOS:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)

_probe = "The appellant filed a petition under Section 9 of the Act."
_ids = tokenizer.encode(_probe)
print(len(_ids), _ids[:20])
print(repr(tokenizer.decode(_ids)))

In [ ]:
print("Document tokens:", length_stats(train_docs, "document"))
print("Summary tokens :", length_stats(train_docs, "summary"))

## 5. Dataset and DataLoaders

**Batch contract** expected by the training loops (sections 12 & 15) and the model (section 8):

*Causal backbones (decoder-only / SSM / CNN-attention)* — one sequence `prompt + summary`:

| key | shape | notes |
|-----|-------|-------|
| `input_ids` | `[B, T]` long | `[BOS] + document(truncated/selected) + separator + summary + [EOS]`, right-padded with `PAD_ID` |
| `attention_mask` | `[B, T]` long | 1 = real token |
| `labels` | `[B, T]` long | = `input_ids` on summary (+EOS) positions, `-100` on prompt and padding (the model shifts internally) |
| `ids` | `List[str]` | not moved to device |

*Encoder-decoder backbone (`encdec`)* — source and target are separate sequences:

| key | shape | notes |
|-----|-------|-------|
| `input_ids` / `attention_mask` | `[B, S]` | document tokens (+ `EOS`), `S ≤ 1024` (`ENCDEC_CONFIG.max_input_length`) |
| `labels` | `[B, T]` long | `summary + [EOS]`, `-100` on padding, `T ≤ 1024` |
| `decoder_input_ids` | optional | if omitted, the Phase-1 model builds it with `shift_right(labels)` (`decoder_start_token_id = PAD_ID`) |

Length budgets: causal backbones need `1 + MAX_SOURCE_TOKENS + separator + MAX_TARGET_TOKENS ≤ 2048`;
`encdec` needs source ≤ 1024 and target ≤ 1024 (checked by `check_length_budget()` in section 11).

The formulation switches (`ENCODER_DECODER`, `STANDARD_ENCDEC`) are set in section 2.

In [ ]:
def build_prompt_ids(document, tokenizer, max_source_tokens=MAX_SOURCE_TOKENS):
    """
    Causal-LM prompt for one document: [BOS] + selected document tokens + SUMMARY_PROMPT tokens.
    Used IDENTICALLY in training (SummDataset) and generation (Section 6 / 12).
    """
    doc_ids = tokenizer.encode(document)
    sep_ids = tokenizer.encode(SUMMARY_PROMPT)
    return [BOS_ID] + doc_ids[:max_source_tokens] + sep_ids


def chunk_document(ids, chunk_size, overlap=CHUNK_OVERLAP):
    """
    Split a tokenized document into overlapping chunks.
    Every part of the document is covered.
    """
    assert 0 <= overlap < chunk_size
    step = chunk_size - overlap

    chunks = []
    for i in range(0, len(ids), step):
        chunk = ids[i:i + chunk_size]
        if chunk:
            chunks.append(chunk)
        if i + chunk_size >= len(ids):
            break

    return chunks if chunks else [[]]


class SummDataset(Dataset):
    """
    One item = one (document, summary) pair.

    __getitem__ returns (causal formulation):
        {"id": str, "prompt_ids": List[int], "target_ids": List[int]}   # target ends with EOS
    Encoder-decoder (encoder_decoder=True):
        {"id": str, "source_ids": List[int], "target_ids": List[int]}   # source <= 1024, target ends with EOS
    """

    def __init__(
        self,
        docs,
        tokenizer,
        max_source_tokens=MAX_SOURCE_TOKENS,
        max_target_tokens=MAX_TARGET_TOKENS,
        encoder_decoder=False,
    ):
        self.items = []
        self.encoder_decoder = encoder_decoder

        for doc in docs:
            doc_ids = tokenizer.encode(doc["document"])
            # Tokenize the complete reference summary.
            summary_ids = tokenizer.encode(doc["summary"])
            
            # Append EOS only when the full reference summary fits.
            # If truncated, keep the last position for a real summary token.
            if len(summary_ids) + 1 <= max_target_tokens:
                target_ids = summary_ids + [EOS_ID]
            else:
                target_ids = summary_ids[:max_target_tokens]

            if encoder_decoder and STANDARD_ENCDEC:
                # Standard formulation: preserve pretrained token-level cross-attention.
                self.items.append({
                    "id": doc["id"],
                    "source_ids": doc_ids[:max_source_tokens] + [EOS_ID],
                    "target_ids": target_ids
                })

            elif encoder_decoder:
                # Hierarchical formulation: keep all document chunks.
                assert max_source_tokens + 1 <= 1024

                chunks = chunk_document(
                    doc_ids,
                    chunk_size=max_source_tokens,
                    overlap=CHUNK_OVERLAP
                )

                # Add EOS independently to every source chunk.
                chunks = [chunk + [EOS_ID] for chunk in chunks]

                self.items.append({
                    "id": doc["id"],
                    "chunks": chunks,
                    "target_ids": target_ids
                })

            else:
                # Standard causal formulation: prompt + target.
                prompt_ids = build_prompt_ids(
                    doc["document"],
                    tokenizer,
                    max_source_tokens
                )

                self.items.append({
                    "id": doc["id"],
                    "prompt_ids": prompt_ids,
                    "target_ids": target_ids
                })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def summ_collator(batch):
    """
    Causal: concatenate prompt + target, right-pad to the batch max length,
    labels = -100 on prompt + padding, token id on target positions.
    Encoder-decoder: pad source with PAD_ID (attention_mask 0) and labels with -100;
    decoder_input_ids may be omitted (the Phase-1 encdec builds it with shift_right(labels)).
    Must return the dict described in the batch-contract table above.
    """

    if "source_ids" in batch[0]:
        # Standard encoder-decoder batch: [B, S] source and [B, T] labels.
        B = len(batch)
        S = max(len(x["source_ids"]) for x in batch)
        T = max(len(x["target_ids"]) for x in batch)
        input_ids = torch.full((B, S), PAD_ID, dtype=torch.long)
        attention_mask = torch.zeros((B, S), dtype=torch.long)
        labels = torch.full((B, T), -100, dtype=torch.long)
        for i, x in enumerate(batch):
            src, tgt = x["source_ids"], x["target_ids"]
            input_ids[i, :len(src)] = torch.tensor(src, dtype=torch.long)
            attention_mask[i, :len(src)] = 1
            labels[i, :len(tgt)] = torch.tensor(tgt, dtype=torch.long)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "ids": [x["id"] for x in batch]
        }

    if "chunks" in batch[0]:
        # Hierarchical encoder-decoder batch.
        # input_ids: [B, C, S]
        # attention_mask: [B, C, S]
        # chunk_mask: [B, C]
        # labels: [B, T]

        B = len(batch)
        C = max(len(x["chunks"]) for x in batch)
        S = max(len(c) for x in batch for c in x["chunks"])
        T = max(len(x["target_ids"]) for x in batch)

        input_ids = torch.full((B, C, S), PAD_ID, dtype=torch.long)
        attention_mask = torch.zeros((B, C, S), dtype=torch.long)
        chunk_mask = torch.zeros((B, C), dtype=torch.long)
        labels = torch.full((B, T), -100, dtype=torch.long)

        for i, x in enumerate(batch):
            # Copy real chunks and mark valid chunk positions.
            for j, chunk in enumerate(x["chunks"]):
                n = len(chunk)

                input_ids[i, j, :n] = torch.tensor(chunk, dtype=torch.long)
                attention_mask[i, j, :n] = 1
                chunk_mask[i, j] = 1

            # Only the document-level summary is supervised.
            target = x["target_ids"]
            labels[i, :len(target)] = torch.tensor(target, dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "chunk_mask": chunk_mask,
            "labels": labels,
            "ids": [x["id"] for x in batch]
        }

    # Standard causal-LM batch.
    # input_ids: [B, T]
    # attention_mask: [B, T]
    # labels: [B, T]

    B = len(batch)
    T = max(len(x["prompt_ids"]) + len(x["target_ids"]) for x in batch)

    input_ids = torch.full((B, T), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((B, T), dtype=torch.long)
    labels = torch.full((B, T), -100, dtype=torch.long)

    for i, x in enumerate(batch):
        prompt = x["prompt_ids"]
        target = x["target_ids"]

        seq = prompt + target
        n = len(seq)

        input_ids[i, :n] = torch.tensor(seq, dtype=torch.long)
        attention_mask[i, :n] = 1

        # Ignore prompt and padding positions in the loss.
        labels[i, len(prompt):n] = torch.tensor(target, dtype=torch.long)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "ids": [x["id"] for x in batch]
    }

In [ ]:
# ============================================================
# DataLoaders (boilerplate) — test documents are summarised by generation (Section 12), not via a loader
# ============================================================

def build_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, encoder_decoder=ENCODER_DECODER):
    # num_workers=0: items are pre-tokenized in __init__; avoids the worker-shutdown warnings seen in Phase 3
    train_ds = SummDataset(train_docs, tokenizer, encoder_decoder=encoder_decoder)
    val_ds   = SummDataset(val_docs, tokenizer, encoder_decoder=encoder_decoder)

    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=summ_collator, num_workers=num_workers, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=summ_collator, num_workers=num_workers)
    return train_ds, val_ds, train_loader, val_loader



train_ds, val_ds, train_loader, val_loader = build_dataloaders()

print("Train items:", len(train_ds))
print("Val items  :", len(val_ds))

_b = next(iter(val_loader))
for k, v in _b.items():
    print(f"{k:18s}", tuple(v.shape) if torch.is_tensor(v) else type(v).__name__)

# Verify standard or hierarchical batch structure.
if ENCODER_DECODER and not STANDARD_ENCDEC:
    assert _b["input_ids"].ndim == 3
    assert _b["chunk_mask"].shape == _b["input_ids"].shape[:2]
else:
    assert _b["input_ids"].ndim == 2
    assert "chunk_mask" not in _b

assert _b["attention_mask"].shape == _b["input_ids"].shape
assert _b["labels"].ndim == 2
assert len(_b["ids"]) == _b["input_ids"].shape[0]

print("DataLoader checks passed.")

In [ ]:
# ============================================================
# Verify Phase 6 summary truncation and EOS handling
# ============================================================

print("Dataset target default:", SummDataset.__init__.__defaults__)

truncated = 0
complete = 0

for doc, item in zip(train_docs, train_ds.items):
    full_len = len(tokenizer.encode(doc["summary"]))
    target = item["target_ids"]

    assert len(target) <= MAX_TARGET_TOKENS

    if full_len + 1 <= MAX_TARGET_TOKENS:
        assert target == tokenizer.encode(doc["summary"]) + [EOS_ID]
        complete += 1
    else:
        assert target == tokenizer.encode(doc["summary"])[:MAX_TARGET_TOKENS]
        truncated += 1

print("Complete summaries:", complete)
print("Truncated summaries:", truncated)
print("Total:", complete + truncated)

## 6. Phase-1 backbone definitions

Every backbone exposes `encode(input_ids, attention_mask=None) -> [B, L, D]` and a `forward(...)` that returns LM `logits`
(used for training and generation). Checkpoint paths are set in section 2.

### 6A. Decoder-only Transformer

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class DecoderConfig:
    vocab_size: int
    context_length: int
    n_embd: int
    n_layer: int
    n_head: int
    embd_pdrop: float = 0.1
    resid_pdrop: float = 0.1
    attn_pdrop: float = 0.1
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    bos_token_id: int = 2
    eos_token_id: int = 3
    pad_token_id: int = 0
    n_inner: int | None = None
    activation_function: str = "gelu_new"
    scale_attn_weights: bool = True
    scale_attn_by_inverse_layer_idx: bool = False
    tie_word_embeddings: bool = True

    @property
    def feed_forward_size(self):
        return self.n_inner if self.n_inner is not None else 4 * self.n_embd


class CausalSelfAttention(nn.Module):
    """Causal multi-head self-attention with a fused QKV projection."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        assert config.n_embd % config.n_head == 0, (
            "n_embd must be divisible by n_head"
        )
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.layer_idx = layer_idx
        self.scale_attn_weights = config.scale_attn_weights
        self.scale_attn_by_inverse_layer_idx = (
            config.scale_attn_by_inverse_layer_idx
        )

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=True)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=True)
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

    def forward(self, hidden_states, attention_mask=None):
            batch_size, seq_len, hidden_size = hidden_states.shape
        
            qkv = self.qkv(hidden_states)
            query, key, value = qkv.split(hidden_size, dim=-1)
        
            def split_heads(tensor):
                return tensor.view(
                    batch_size,
                    seq_len,
                    self.n_head,
                    self.head_dim
                ).transpose(1, 2)
        
            query = split_heads(query)
            key = split_heads(key)
            value = split_heads(value)
        
            scale = None
        
            if self.scale_attn_weights:
                scale = 1.0 / math.sqrt(self.head_dim)
        
            if self.scale_attn_by_inverse_layer_idx:
                if scale is None:
                    scale = 1.0
                scale = scale / float(self.layer_idx + 1)
        
            attn_mask = None
            is_causal = True
        
            if attention_mask is not None:
                key_is_valid = attention_mask.to(
                    device=hidden_states.device,
                    dtype=torch.bool
                )
        
                # [B, 1, 1, L]
                attn_mask = key_is_valid[:, None, None, :]
        
                # Since an explicit mask is supplied, combine causal mask manually
                causal = torch.ones(
                    seq_len,
                    seq_len,
                    dtype=torch.bool,
                    device=hidden_states.device
                ).tril()
        
                attn_mask = attn_mask & causal[None, None, :, :]
                is_causal = False
        
            attended = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=attn_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
                scale=scale,
            )
        
            attended = attended.transpose(1, 2).contiguous().view(
                batch_size,
                seq_len,
                hidden_size
            )
        
            return self.resid_dropout(
                self.out_proj(attended)
            )


class DecoderOnlyMLP(nn.Module):
    """Position-wise feed-forward block (expand -> activation -> project)."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        self.fc_in = nn.Linear(
            config.n_embd, config.feed_forward_size, bias=True
        )
        self.fc_out = nn.Linear(
            config.feed_forward_size, config.n_embd, bias=True
        )
        self.dropout = nn.Dropout(config.resid_pdrop)
        self.activation_name = config.activation_function

    def _activation(self, x):
        if self.activation_name == "gelu_new":
            # Tanh approximation of GELU.
            return 0.5 * x * (
                1.0
                + torch.tanh(
                    math.sqrt(2.0 / math.pi)
                    * (x + 0.044715 * torch.pow(x, 3.0))
                )
            )
        if self.activation_name == "gelu":
            return F.gelu(x)
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported activation: {self.activation_name}"
        )

    def forward(self, hidden_states):
        return self.dropout(self.fc_out(self._activation(self.fc_in(hidden_states))))


class DecoderOnlyBlock(nn.Module):
    """Pre-norm Transformer block with residual attention and MLP paths."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.attn = CausalSelfAttention(config, layer_idx)
        self.ln_2 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.mlp = DecoderOnlyMLP(config)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = hidden_states + self.attn(
            self.ln_1(hidden_states), attention_mask=attention_mask
        )
        hidden_states = hidden_states + self.mlp(self.ln_2(hidden_states))
        return hidden_states


class DecoderOnlyTransformer(nn.Module):
    """Decoder-only causal language model implemented in raw PyTorch."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        if config.n_embd % config.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if config.context_length <= 0:
            raise ValueError("context_length must be positive")
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(
            config.context_length, config.n_embd
        )
        self.embedding_dropout = nn.Dropout(config.embd_pdrop)
        self.blocks = nn.ModuleList(
            DecoderOnlyBlock(config, layer_idx)
            for layer_idx in range(config.n_layer)
        )
        self.final_ln = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._initialize_weights)
        if config.tie_word_embeddings:
            # Weight tying: the output projection shares the token-embedding matrix.
            self.lm_head.weight = self.token_embedding.weight

    def _initialize_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def _validate_inputs(self, input_ids, attention_mask=None):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [batch, sequence]")
        batch_size, seq_len = input_ids.shape
        if seq_len == 0:
            raise ValueError("input sequence must not be empty")
        assert seq_len <= self.config.context_length, (
            f"sequence length {seq_len} exceeds context length "
            f"{self.config.context_length}"
        )
        if attention_mask is not None:
            if attention_mask.shape != (batch_size, seq_len):
                raise ValueError(
                    "attention_mask must have shape [batch, sequence]"
                )
        return batch_size, seq_len

    def _embed(self, input_ids):
        _, seq_len = input_ids.shape
        position_ids = torch.arange(
            seq_len, dtype=torch.long, device=input_ids.device
        ).unsqueeze(0)
        return self.embedding_dropout(
            self.token_embedding(input_ids)
            + self.position_embedding(position_ids)
        )

    def encode(self, input_ids, attention_mask=None):
        """Return final normalized hidden states with shape [B, L, D]."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        for block in self.blocks:
            hidden_states = block(hidden_states, attention_mask)
        return self.final_ln(hidden_states)

    def debug_hidden_states(self, input_ids, attention_mask=None):
        """Expose intermediate states for optional migration debugging."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            hidden_states = block(hidden_states, attention_mask)
            states[f"block_{layer_idx}"] = hidden_states
        states["final_norm"] = self.final_ln(hidden_states)
        return states

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.encode(input_ids, attention_mask)
        logits = self.lm_head(hidden_states)
        loss = None

        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError("labels must have the same shape as input_ids")
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            valid_labels = shift_labels.ne(-100)
            if attention_mask is not None:
                valid_labels = valid_labels & attention_mask[:, 1:].bool()
            
            shift_labels = shift_labels.masked_fill(~valid_labels, -100)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
            "hidden_states": hidden_states,
        }

In [ ]:
decoder_config = DecoderConfig(
    vocab_size=16000,
    context_length=2048,
    n_embd=640,
    n_layer=8,
    n_head=8,
    embd_pdrop=0.1,
    resid_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    n_inner=None,
    activation_function="gelu_new",
    scale_attn_weights=True,
    scale_attn_by_inverse_layer_idx=False,
    tie_word_embeddings=True,
)

### 6B. Diagonal SSM / S4D-inspired LM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SSMLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim,
        n_layer,
        max_seq_len,
        state_dim=16,
        expand=3,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim
        self.n_layer = n_layer
        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(
            vocab_size,
            dim
        )

        self.layers = nn.ModuleList([
            SSMBlock(
                dim=dim,
                state_dim=state_dim,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.norm_f = nn.LayerNorm(dim)

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False
        )

    def forward(
        self,
        input_ids,
        labels=None,
        **kwargs
    ):
        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.max_seq_len}"
            )

        x = self.embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_f(x)

        logits = self.lm_head(x)

        if labels is not None:

            targets = torch.full_like(
                labels,
                -100
            )

            targets[:, :-1] = labels[:, 1:]

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    self.vocab_size
                ),
                targets.reshape(-1),
                ignore_index=-100,
            )

            return {
                "loss": loss
            }

        return {
            "logits": logits
        }
class SSMBlock(nn.Module):
    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.ssm = DiagonalSSM(
            dim=dim,
            state_dim=state_dim,
            expand=expand,
        )

    def forward(self, x):
        return x + self.ssm(self.norm(x))
class DiagonalSSM(nn.Module):
    """
    S4D-inspired diagonal state-space layer.

    Input : [B, T, C]
    Output: [B, T, C]
    """

    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.dim = dim
        self.state_dim = state_dim
        self.inner_dim = dim * expand

        # content + gate
        self.in_proj = nn.Linear(
            dim,
            2 * self.inner_dim
        )

        # Stable diagonal state transition
        # A will become negative
        A = torch.arange(
            1,
            state_dim + 1,
            dtype=torch.float32
        )

        self.A_log = nn.Parameter(
            torch.log(A)
        )

        # Per-channel SSM input/output weights
        self.B = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        self.C = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        # Per-channel time scale
        self.log_dt = nn.Parameter(
            torch.zeros(self.inner_dim)
        )

        # Direct skip term
        self.D = nn.Parameter(
            torch.ones(self.inner_dim)
        )

        self.out_proj = nn.Linear(
            self.inner_dim,
            dim
        )

    def forward(self, x):
        Bsz, T, _ = x.shape

        xz = self.in_proj(x)

        u, z = xz.chunk(
            2,
            dim=-1
        )

        # -------------------------------------------------
        # Step 1
        #
        # Construct stable continuous transition:
        #
        # A = -exp(A_log)
        #
        # shape:
        # [state_dim]
        # -------------------------------------------------

        A = -torch.exp(self.A_log)

        # -------------------------------------------------
        # Step 2
        #
        # dt must always be positive.
        #
        # use softplus(log_dt)
        #
        # shape:
        # [inner_dim]
        # -------------------------------------------------

        dt = F.softplus(self.log_dt)

        # -------------------------------------------------
        # Step 3
        #
        # Create discrete transition:
        #
        # A_bar[c,n] = exp(dt[c] * A[n])
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        A_bar = torch.exp(
            dt.unsqueeze(-1) * A.unsqueeze(0)
        )

        # -------------------------------------------------
        # Step 4
        #
        # Zero-order-hold style input coefficient:
        #
        # B_bar[c,n]
        #   = ((A_bar[c,n] - 1) / A[n]) * B[c,n]
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        xA = dt.unsqueeze(-1) * A.unsqueeze(0)

        B_bar = (
            torch.expm1(xA) / A.unsqueeze(0)
        ) * self.B

        # -------------------------------------------------
        # Step 5
        #
        # Build time indices:
        #
        # t = [0, 1, ..., T-1]
        #
        # Then construct the impulse-response kernel:
        #
        # K[c,t] =
        #   sum_n C[c,n]
        #         * A_bar[c,n]^t
        #         * B_bar[c,n]
        #
        # Target shape:
        # [inner_dim, T]
        #
        # Hint:
        # avoid a Python loop over T.
        # Broadcasting / einsum should be used.
        # -------------------------------------------------

        t = torch.arange(
            T,
            device=x.device,
            dtype=A_bar.dtype
        )

        powers = A_bar.unsqueeze(-1) ** t
        # [inner_dim, state_dim, T]

        kernel = torch.einsum(
            "cn,cnt->ct",
            self.C * B_bar,
            powers
        )
        # [inner_dim, T]

        # -------------------------------------------------
        # Step 6 — FFT causal convolution
        #
        # u currently:
        # [B, T, inner_dim]
        #
        # transpose to:
        # [B, inner_dim, T]
        #
        # Use an FFT length >= 2*T.
        #
        # rfft(input)
        # rfft(kernel)
        # multiply
        # irfft
        #
        # finally truncate to first T values.
        #
        # This gives causal convolution:
        #
        # y[t] = sum_{i=0}^{t} K[i] u[t-i]
        # -------------------------------------------------

        u = u.transpose(1, 2)
        # [B, inner_dim, T]

        fft_len = 2 * T

        u_fft = u.float()
        kernel_fft = kernel.float()
        
        u_f = torch.fft.rfft(
            u_fft,
            n=fft_len
        )
        
        k_f = torch.fft.rfft(
            kernel_fft,
            n=fft_len
        )
        
        y = torch.fft.irfft(
            u_f * k_f.unsqueeze(0),
            n=fft_len
        )[..., :T]
        
        y = y.to(u.dtype)

        # -------------------------------------------------
        # Step 7
        #
        # Add direct SSM skip:
        #
        # y += D * u
        # -------------------------------------------------

        y = y + self.D.view(1, -1, 1) * u

        # restore:
        # [B, T, inner_dim]

        y = y.transpose(1, 2)

        # same gating idea as before
        y = y * F.silu(z)

        return self.out_proj(y)

In [ ]:
def ssm_encode(self, input_ids, attention_mask=None):
    B, T = input_ids.shape

    if T > self.max_seq_len:
        raise ValueError(
            f"Sequence length {T} exceeds max_seq_len={self.max_seq_len}"
        )

    x = self.embedding(input_ids)

    for layer in self.layers:
        x = layer(x)

    x = self.norm_f(x)

    return x

SSMLM.encode = ssm_encode

In [ ]:
SSM_DIM = 700
SSM_N_LAYER = 10
SSM_MAX_SEQ_LEN = 2048
SSM_STATE_DIM = 16
SSM_EXPAND = 3


def build_ssm_backbone():
    return SSMLM(
        vocab_size=16000,
        dim=SSM_DIM,
        n_layer=SSM_N_LAYER,
        max_seq_len=SSM_MAX_SEQ_LEN,
        state_dim=SSM_STATE_DIM,
        expand=SSM_EXPAND,
    )

### 6C. Encoder-decoder Transformer

`document → encoder`, `summary → decoder` (`decoder_start_token_id = PAD_ID`, `shift_right` built in). Inputs ≤ 1024 tokens.

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class EncoderDecoderConfig:
    vocab_size: int
    d_model: int
    d_kv: int
    d_ff: int
    num_layers: int
    num_decoder_layers: int
    num_heads: int
    dropout_rate: float = 0.1
    layer_norm_epsilon: float = 1e-6
    relative_attention_num_buckets: int = 32
    relative_attention_max_distance: int = 128
    feed_forward_proj: str = "relu"
    dense_act_fn: str = "relu"
    tie_word_embeddings: bool = True
    scale_decoder_outputs: bool = True
    initializer_factor: float = 1.0
    pad_token_id: int = 0
    eos_token_id: int = 3
    decoder_start_token_id: int = 0
    max_input_length: int = 1024
    max_target_length: int = 1024
    use_cache: bool = False

    @property
    def inner_dim(self):
        return self.num_heads * self.d_kv

    @property
    def is_gated_act(self):
        return self.feed_forward_proj.startswith("gated-")


class RMSNorm(nn.Module):
    """Scale-only RMS normalization (no mean subtraction, no bias)."""

    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        variance = hidden_states.float().pow(2).mean(dim=-1, keepdim=True)
        normalized = hidden_states * torch.rsqrt(
            variance + self.variance_epsilon
        ).to(dtype=hidden_states.dtype)
        return normalized * self.weight.to(dtype=hidden_states.dtype)


class RelativePositionBias(nn.Module):
    """Bucketed relative-position bias: log-spaced buckets, one learned bias per head."""

    def __init__(
        self,
        num_buckets,
        num_heads,
        max_distance,
        bidirectional,
    ):
        super().__init__()
        if num_buckets < 2:
            raise ValueError("relative_attention_num_buckets must be >= 2")
        if max_distance < 2:
            raise ValueError("relative_attention_max_distance must be >= 2")
        self.num_buckets = num_buckets
        self.num_heads = num_heads
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def relative_position_bucket(
        relative_position,
        bidirectional=True,
        num_buckets=32,
        max_distance=128,
    ):
        """Map key_position - query_position to a relative-position bucket."""
        bucket = 0
        if bidirectional:
            num_buckets //= 2
            bucket = bucket + (
                (relative_position > 0).to(torch.long) * num_buckets
            )
            relative_position = torch.abs(relative_position)
        else:
            relative_position = -torch.minimum(
                relative_position, torch.zeros_like(relative_position)
            )

        max_exact = num_buckets // 2
        is_small = relative_position < max_exact
        relative_position_if_large = max_exact + (
            torch.log(
                relative_position.float().clamp(min=1) / max_exact
            )
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        relative_position_if_large = torch.clamp(
            relative_position_if_large, max=num_buckets - 1
        )
        bucket = bucket + torch.where(
            is_small, relative_position, relative_position_if_large
        )
        return bucket

    def forward(self, query_length, key_length, device=None):
        if device is None:
            device = self.embedding.weight.device
        query_positions = torch.arange(
            query_length, dtype=torch.long, device=device
        )[:, None]
        key_positions = torch.arange(
            key_length, dtype=torch.long, device=device
        )[None, :]
        relative_position = key_positions - query_positions
        buckets = self.relative_position_bucket(
            relative_position,
            bidirectional=self.bidirectional,
            num_buckets=self.num_buckets,
            max_distance=self.max_distance,
        )
        values = self.embedding(buckets)
        return values.permute(2, 0, 1).unsqueeze(0)


class MultiHeadAttention(nn.Module):
    """Multi-head attention with bias-free Q/K/V/O projections."""

    def __init__(
        self,
        config: EncoderDecoderConfig,
        causal=False,
        relative_bias_bidirectional=None,
    ):
        super().__init__()
        self.d_model = config.d_model
        self.d_kv = config.d_kv
        self.num_heads = config.num_heads
        self.inner_dim = config.inner_dim
        self.causal = causal

        self.q = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.k = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.v = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.o = nn.Linear(self.inner_dim, self.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.relative_attention_bias = None
        if relative_bias_bidirectional is not None:
            self.relative_attention_bias = RelativePositionBias(
                num_buckets=config.relative_attention_num_buckets,
                num_heads=config.num_heads,
                max_distance=config.relative_attention_max_distance,
                bidirectional=relative_bias_bidirectional,
            )

    def _split_heads(self, projected):
        batch_size, seq_len, _ = projected.shape
        return projected.view(
            batch_size, seq_len, self.num_heads, self.d_kv
        ).transpose(1, 2)

    def forward(
        self,
        hidden_states,
        key_value_states=None,
        attention_mask=None,
        position_bias=None,
    ):
        if key_value_states is None:
            key_value_states = hidden_states
        query = self._split_heads(self.q(hidden_states))
        key = self._split_heads(self.k(key_value_states))
        value = self._split_heads(self.v(key_value_states))

        scores = torch.matmul(query, key.transpose(-2, -1))
        query_length, key_length = scores.shape[-2:]

        if position_bias is None and self.relative_attention_bias is not None:
            position_bias = self.relative_attention_bias(
                query_length, key_length, device=scores.device
            )
        if position_bias is not None:
            scores = scores + position_bias.to(dtype=scores.dtype)

        if self.causal:
            if query_length != key_length:
                raise ValueError(
                    "This implementation does not use a KV cache; "
                    "causal self-attention requires equal query/key lengths."
                )
            causal_allowed = torch.ones(
                query_length,
                key_length,
                dtype=torch.bool,
                device=scores.device,
            ).tril()
            scores = scores.masked_fill(
                ~causal_allowed.view(1, 1, query_length, key_length),
                torch.finfo(scores.dtype).min,
            )

        if attention_mask is not None:
            if attention_mask.ndim != 2 or attention_mask.shape[-1] != key_length:
                raise ValueError(
                    "attention_mask must have shape [batch, key_length]"
                )
            key_is_valid = attention_mask.to(
                device=scores.device, dtype=torch.bool
            ).view(scores.shape[0], 1, 1, key_length)
            scores = scores.masked_fill(
                ~key_is_valid, torch.finfo(scores.dtype).min
            )

        probabilities = F.softmax(scores.float(), dim=-1).to(
            dtype=scores.dtype
        )
        probabilities = self.dropout(probabilities)
        attended = torch.matmul(probabilities, value)
        attended = attended.transpose(1, 2).contiguous().view(
            attended.shape[0], query_length, self.inner_dim
        )
        return self.o(attended)


class EncoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=False,
            relative_bias_bidirectional=(
                True if has_relative_attention_bias else None
            ),
        )


class DecoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=True,
            relative_bias_bidirectional=(
                False if has_relative_attention_bias else None
            ),
        )


class CrossAttention(MultiHeadAttention):
    def __init__(self, config):
        super().__init__(config, causal=False)


class EncDecFeedForward(nn.Module):
    """Feed-forward layer: dense-act-dense, or gated (two input projections)."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.feed_forward_proj = config.feed_forward_proj
        self.activation_name = config.dense_act_fn
        if config.is_gated_act:
            self.wi_0 = nn.Linear(config.d_model, config.d_ff, bias=False)
            self.wi_1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        else:
            self.wi = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.wo = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _activation(self, x):
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("gelu", "gelu_new"):
            return F.gelu(x, approximate="tanh" if self.activation_name == "gelu_new" else "none")
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported dense_act_fn: {self.activation_name}"
        )

    def forward(self, hidden_states):
        if self.feed_forward_proj.startswith("gated-"):
            hidden_states = self._activation(self.wi_0(hidden_states)) * self.wi_1(
                hidden_states
            )
        else:
            hidden_states = self._activation(self.wi(hidden_states))
        hidden_states = self.dropout(hidden_states)
        return self.wo(hidden_states)


class EncoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = EncoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self, hidden_states, attention_mask=None, position_bias=None,
        return_debug=False
    ):
        debug = {}
        attention_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(attention_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class DecoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = DecoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.cross_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.cross_attention = CrossAttention(config)
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self,
        hidden_states,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        position_bias=None,
        return_debug=False,
    ):
        debug = {}
        self_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=decoder_attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(self_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        cross_output = self.cross_attention(
            self.cross_norm(hidden_states),
            key_value_states=encoder_hidden_states,
            attention_mask=encoder_attention_mask,
        )
        hidden_states = hidden_states + self.dropout(cross_output)
        if return_debug:
            debug["cross_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class Encoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            EncoderBlock(config, layer_idx)
            for layer_idx in range(config.num_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(self, inputs_embeds, attention_mask=None, debug=False):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                states[f"block_{layer_idx}_self_attention"] = (
                    block_debug["self_attention_output"]
                )
                states[f"block_{layer_idx}_feed_forward"] = (
                    block_debug["feed_forward_output"]
                )
            else:
                hidden_states = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=False)

    def debug_hidden_states(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=True)


class Decoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            DecoderBlock(config, layer_idx)
            for layer_idx in range(config.num_decoder_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        debug=False,
    ):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                for name, value in block_debug.items():
                    states[f"block_{layer_idx}_{name}"] = value
            else:
                hidden_states = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=False,
        )

    def debug_hidden_states(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=True,
        )


class EncoderDecoderTransformer(nn.Module):
    """Encoder-decoder Transformer, implemented with raw PyTorch layers."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        if config.d_model <= 0 or config.d_kv <= 0:
            raise ValueError("d_model and d_kv must be positive")
        if config.num_heads <= 0 or config.num_layers <= 0:
            raise ValueError("num_heads and num_layers must be positive")
        if config.num_decoder_layers <= 0:
            raise ValueError("num_decoder_layers must be positive")
        self.config = config
        self.shared_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self._initialize_weights()
        if config.tie_word_embeddings:
            # One embedding matrix is shared by the encoder, decoder and LM head.
            self.lm_head.weight = self.shared_embedding.weight

    def _initialize_weights(self):
        factor = self.config.initializer_factor
        with torch.no_grad():
            self.shared_embedding.weight.normal_(mean=0.0, std=factor)
            for module in self.modules():
                if isinstance(module, RMSNorm):
                    module.weight.fill_(1.0)
                elif isinstance(module, RelativePositionBias):
                    module.embedding.weight.normal_(mean=0.0, std=factor)
                elif isinstance(module, MultiHeadAttention):
                    module.q.weight.normal_(
                        mean=0.0,
                        std=factor
                        * (self.config.d_model * self.config.d_kv) ** -0.5,
                    )
                    for projection in (module.k, module.v):
                        projection.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.o.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.inner_dim ** -0.5,
                    )
                elif isinstance(module, EncDecFeedForward):
                    if self.config.is_gated_act:
                        module.wi_0.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                        module.wi_1.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    else:
                        module.wi.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.wo.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.d_ff ** -0.5,
                    )

    def _validate_input_ids(self, input_ids, max_length, name):
        if input_ids.ndim != 2:
            raise ValueError(f"{name} must have shape [batch, sequence]")
        if input_ids.shape[1] == 0:
            raise ValueError(f"{name} must not be empty")
        if input_ids.shape[1] > max_length:
            raise ValueError(
                f"{name} length {input_ids.shape[1]} exceeds {max_length}"
            )

    def encode(self, input_ids, attention_mask=None):
        """Return final encoder states with shape [B, L, d_model]."""
        self._validate_input_ids(
            input_ids, self.config.max_input_length, "input_ids"
        )
        if attention_mask is not None and attention_mask.shape != input_ids.shape:
            raise ValueError("attention_mask must match input_ids shape")
        return self.encoder(
            self.shared_embedding(input_ids),
            attention_mask=attention_mask,
        )

    def decode(
        self,
        decoder_input_ids,
        encoder_hidden_states,
        attention_mask=None,
        decoder_attention_mask=None,
    ):
        self._validate_input_ids(
            decoder_input_ids,
            self.config.max_target_length,
            "decoder_input_ids",
        )
        if decoder_input_ids.shape[0] != encoder_hidden_states.shape[0]:
            raise ValueError(
                "decoder and encoder batch dimensions must match"
            )
        return self.decoder(
            self.shared_embedding(decoder_input_ids),
            encoder_hidden_states,
            encoder_attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )

    def shift_right(self, labels):
        if labels.ndim != 2:
            raise ValueError("labels must have shape [batch, target_length]")
        if labels.shape[1] == 0:
            raise ValueError("labels must not be empty")
        shifted = labels.new_full(labels.shape, self.config.pad_token_id)
        shifted[:, 0] = self.config.decoder_start_token_id
        shifted[:, 1:] = labels[:, :-1]
        shifted.masked_fill_(shifted == -100, self.config.pad_token_id)
        return shifted

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        encoder_hidden_states=None,
    ):
        if encoder_hidden_states is None:
            if input_ids is None:
                raise ValueError(
                    "input_ids are required when encoder states are not supplied"
                )
            encoder_hidden_states = self.encode(input_ids, attention_mask)
        elif input_ids is not None:
            self._validate_input_ids(
                input_ids, self.config.max_input_length, "input_ids"
            )

        if decoder_input_ids is None:
            if labels is None:
                raise ValueError(
                    "Pass decoder_input_ids or labels to run the decoder"
                )
            decoder_input_ids = self.shift_right(labels)
        if decoder_input_ids.shape[1] > self.config.max_target_length:
            raise ValueError("decoder_input_ids exceed max_target_length")

        decoder_hidden_states = self.decode(
            decoder_input_ids,
            encoder_hidden_states,
            attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )
        if (
            self.config.tie_word_embeddings
            and self.config.scale_decoder_outputs
        ):
            decoder_hidden_states = decoder_hidden_states * (
                self.config.d_model ** -0.5
            )
        logits = self.lm_head(decoder_hidden_states)
        loss = None
        if labels is not None:
            if labels.shape != logits.shape[:2]:
                raise ValueError(
                    "labels must match the decoder logits sequence dimensions"
                )
            loss_labels = labels.masked_fill(
                labels == self.config.pad_token_id, -100
            )
            loss = F.cross_entropy(
                logits.float().reshape(-1, self.config.vocab_size),
                loss_labels.reshape(-1),
                ignore_index=-100,
            )
        return {
            "loss": loss,
            "logits": logits,
            "encoder_hidden_states": encoder_hidden_states,
        }



# Phase-3 uses the encoder representation from this pretrained Phase-1 model.
def _encoder_decoder_encode(self, input_ids, attention_mask=None):
    return self.encode(input_ids, attention_mask)

# The original Phase-1 class already defines encode(); this alias documents the
# task-facing API without changing its state-dict parameter names.

ENCDEC_CONFIG = EncoderDecoderConfig(
    vocab_size=16000,
    d_model=512,
    d_kv=64,
    d_ff=2048,
    num_layers=6,
    num_decoder_layers=6,
    num_heads=8,
    dropout_rate=0.1,
    layer_norm_epsilon=1e-6,
    relative_attention_num_buckets=32,
    relative_attention_max_distance=128,
    feed_forward_proj="relu",
    dense_act_fn="relu",
    tie_word_embeddings=True,
    scale_decoder_outputs=True,
    initializer_factor=1.0,
    pad_token_id=PAD_ID,
    eos_token_id=EOS_ID,
    decoder_start_token_id=PAD_ID,
    max_input_length=1024,
    max_target_length=1024,
)


def build_encdec_backbone():
    return EncoderDecoderTransformer(ENCDEC_CONFIG)

### 6D. CNN + Causal Self-Attention

Its attention class is named `CNNCausalSelfAttention` so it does not overwrite the decoder's `CausalSelfAttention` (parameter names unchanged).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
#1
class CausalConvBlock(nn.Module):


        def __init__(self, dim, kernel_size):
            super().__init__()
        
            self.kernel_size = kernel_size
        
            self.conv = nn.Conv1d(
                in_channels=dim,
                out_channels=dim,
                kernel_size=kernel_size
            )
        
        def forward(self, x):
        
            # x: [B, T, C]
        
            # Step 1: transpose
            x = x.transpose(1, 2)  # [B, C, T]
        
            # Step 2: left padding only
            x = F.pad(x, (self.kernel_size - 1, 0))
        
            # Step 3: convolution
            x = self.conv(x)
        
            # Step 4: transpose back
            x = x.transpose(1, 2)  # [B, T, C]
        
            return x


import math
class CNNCausalSelfAttention(nn.Module):


    def __init__(self, dim, n_heads):
        super().__init__()
    
        assert dim % n_heads == 0
    
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
    
        # Step 1: Linear projections for Q, K, V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
    
        # Step 2: Final projection
        self.out_proj = nn.Linear(dim, dim)
    
    
    def forward(self, x):
    
        # x: [B, T, C]
        B, T, C = x.shape
    
        # Step 3: Compute Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
    
        # Step 4: Split into heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=True
        )
    
        # Step 10: Merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)
    
        # Step 11: Final projection
        out = self.out_proj(out)
    
        return out


class HybridBlock(nn.Module):

        
        def __init__(self, dim, n_heads, kernel_size, ff_mult=4):
            super().__init__()
        
            # Step 1: LayerNorm before CNN
            self.norm1 = nn.LayerNorm(dim)
        
            # Step 2: Causal CNN block
            self.conv = CausalConvBlock(dim, kernel_size)
        
            # Step 3: LayerNorm before Attention
            self.norm2 = nn.LayerNorm(dim)
        
            # Step 4: Causal Self-Attention
            self.attn = CNNCausalSelfAttention(dim, n_heads)
        
            # Step 5: LayerNorm before FFN
            self.norm3 = nn.LayerNorm(dim)
        
            # Step 6: Feed Forward Network
            self.ffn = nn.Sequential(
                nn.Linear(dim, ff_mult * dim),
                nn.GELU(),
                nn.Linear(ff_mult * dim, dim)
            )
        
        
        def forward(self, x):
        
            # x shape: [B, T, C]
        
            # Step 7: CNN + residual
            x = x + self.conv(self.norm1(x))
        
            # Step 8: Attention + residual
            x = x + self.attn(self.norm2(x))
        
            # Step 9: FFN + residual
            x = x + self.ffn(self.norm3(x))
        
            return x



class CNNAttentionLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim,
        n_layers,
        n_heads,
        max_seq_len,
        kernel_size=3,
        ff_mult=4
    ):
        super().__init__()

        # Save important model settings
        self.vocab_size = vocab_size
        self.dim = dim
        self.max_seq_len = max_seq_len

        # Converts each token ID into a dim-dimensional vector
        # Input shape:  [B, T]
        # Output shape: [B, T, dim]
        self.token_emb = nn.Embedding(vocab_size, dim)

        # Learnable positional embeddings
        # Gives the model information about token position in sequence
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        # Stack multiple CNN + Attention hybrid blocks
        self.blocks = nn.ModuleList([
            HybridBlock(
                dim,
                n_heads,
                kernel_size,
                ff_mult
            )
            for _ in range(n_layers)
        ])

        # Final normalization before vocabulary prediction
        self.final_norm = nn.LayerNorm(dim)

        # Language modeling head
        # Converts hidden representation into logits over vocabulary
        #
        # [B, T, dim] -> [B, T, vocab_size]
        self.lm_head = nn.Linear(dim, vocab_size)


    def forward(self, input_ids, labels=None, **kwargs):

            B, T = input_ids.shape
    
            assert T <= self.max_seq_len
    
            # Token embeddings: [B,T] -> [B,T,C]
            tok = self.token_emb(input_ids)
    
            # Position IDs: [0,1,...,T-1]
            positions = torch.arange(
                T,
                device=input_ids.device
            )
    
            # Position embeddings: [T] -> [T,C]
            pos = self.pos_emb(positions)
    
            # Broadcasting gives [B,T,C]
            x = tok + pos
    
            # CNN + Attention hybrid blocks
            for block in self.blocks:
                x = block(x)
    
            # Final normalization
            x = self.final_norm(x)
    
            # Vocabulary predictions
            # [B,T,C] -> [B,T,V]
            logits = self.lm_head(x)
    
            loss = None
            
            if labels is not None:
            
                # Create targets with the same sequence length
                targets = torch.full_like(
                    labels,
                    -100
                )
            
                # Position t predicts token t+1
                targets[:, :-1] = labels[:, 1:]
            
                # The final position has target -100,
                # so cross entropy ignores it
                loss = F.cross_entropy(
                    logits.reshape(-1, self.vocab_size),
                    targets.reshape(-1),
                    ignore_index=-100
                )
            
                # Trainer only needs loss during training
                return {
                    "loss": loss
                }
            
            # During generation we need logits
            return {
                "logits": logits
            }

# Match the common Phase-1 backbone interface used by Phase 3. The model is
# causal, so right-padding tokens cannot affect earlier sentence token states.
def _cnn_encode(self, input_ids, attention_mask=None):
    batch_size, seq_len = input_ids.shape
    if seq_len > self.max_seq_len:
        raise ValueError(
            f"Sequence length {seq_len} exceeds max_seq_len={self.max_seq_len}"
        )
    token_states = self.token_emb(input_ids)
    position_ids = torch.arange(seq_len, device=input_ids.device)
    hidden_states = token_states + self.pos_emb(position_ids).unsqueeze(0)
    for block in self.blocks:
        hidden_states = block(hidden_states)
    return self.final_norm(hidden_states)

CNNAttentionLM.encode = _cnn_encode
CNN_DIM = 640
CNN_N_LAYERS = 8
CNN_N_HEADS = 8
CNN_MAX_SEQ_LEN = 2048
CNN_KERNEL_SIZE = 3
CNN_FF_MULT = 4

def build_cnn_backbone():
    return CNNAttentionLM(
        vocab_size=16000,
        dim=CNN_DIM,
        n_layers=CNN_N_LAYERS,
        n_heads=CNN_N_HEADS,
        max_seq_len=CNN_MAX_SEQ_LEN,
        kernel_size=CNN_KERNEL_SIZE,
        ff_mult=CNN_FF_MULT,
    )

In [ ]:
# Maximum sequence length accepted by each backbone (chunk / prompt sizes must respect it)
BACKBONE_MAX_TOKENS = {
    "decoder": decoder_config.context_length,       # 2048
    "ssm": SSM_MAX_SEQ_LEN,                          # 2048
    "encdec": ENCDEC_CONFIG.max_input_length,        # 1024 (encoder input)
    "cnn": CNN_MAX_SEQ_LEN,                          # 2048
}
print(BACKBONE_MAX_TOKENS)

### 6E. Auto-regressive decoding

`beam_search()` is architecture-agnostic: it only needs a function returning next-token logits.
- `causal_generate()` — decoder-only / SSM / CNN-attention: re-runs the window each step (no KV cache — simple and correct, but slow; cap lengths).
- `seq2seq_generate()` — encoder-decoder: encodes the document **once**, then decodes from `decoder_start_token_id` (= `PAD_ID`) with the cached encoder states.

In [ ]:
@torch.no_grad()
def beam_search(next_logits_fn, start_tokens, max_new_tokens, eos_id,
                num_beams=1, length_penalty=1.0, banned_ids=(), device=DEVICE):
    """
    next_logits_fn : callable(LongTensor [N, T]) -> logits [N, V] for the NEXT token of each row
    start_tokens   : List[int] fed before generation (causal prompt, or [decoder_start] for seq2seq)
    num_beams      : 1 = greedy
    length_penalty : final score = sum_logprob / (generated_len ** length_penalty)
    banned_ids     : token ids never generated (e.g. PAD, BOS)
    Returns List[int] generated ids (EOS stripped).
    """
    assert num_beams >= 1 and max_new_tokens >= 1
    start_tokens = list(start_tokens)
    n_start = len(start_tokens)
    banned = torch.tensor(list(banned_ids), dtype=torch.long, device=device) if banned_ids else None

    beams = [(start_tokens, 0.0)]          # alive: (tokens, sum logprob)
    finished = []                          # (tokens incl. EOS, sum logprob)

    def final_score(tokens, logp):
        return logp / (max(len(tokens) - n_start, 1) ** length_penalty)

    for _ in range(max_new_tokens):
        x = torch.tensor([b[0] for b in beams], dtype=torch.long, device=device)   # equal lengths
        logits = next_logits_fn(x).float()                                         # [N, V]
        if banned is not None:
            logits[:, banned] = float("-inf")
        logp = F.log_softmax(logits, dim=-1)

        V = logp.size(-1)
        cand = (torch.tensor([b[1] for b in beams], device=device).unsqueeze(1) + logp).view(-1)
        top_scores, top_idx = cand.topk(min(2 * num_beams, cand.numel()))

        new_beams = []
        for score, idx in zip(top_scores.tolist(), top_idx.tolist()):
            b, tok = divmod(idx, V)
            seq = beams[b][0] + [tok]
            if tok == eos_id:
                finished.append((seq, score))
            else:
                new_beams.append((seq, score))
            if len(new_beams) == num_beams:
                break
        beams = new_beams

        if len(finished) >= num_beams or not beams:
            break

    pool = finished if finished else beams
    best_tokens, _ = max(pool, key=lambda t: final_score(*t))
    out = best_tokens[n_start:]
    if out and out[-1] == eos_id:
        out = out[:-1]
    return out


@torch.no_grad()
def causal_generate(lm, prompt_ids, max_new_tokens, eos_id=None, num_beams=1, length_penalty=1.0,
                    context_length=2048, banned_ids=None, device=DEVICE):
    """lm: causal Phase-1 backbone (raw or PEFT-wrapped); lm(input_ids=..., attention_mask=...)["logits"] -> [N, T, V]."""
    eos_id = EOS_ID if eos_id is None else eos_id
    banned_ids = (PAD_ID, BOS_ID) if banned_ids is None else banned_ids

    def next_logits(x):
        x = x[:, -context_length:]                       # slide if prompt + generated exceeds the window
        return lm(input_ids=x, attention_mask=torch.ones_like(x))["logits"][:, -1, :]

    return beam_search(next_logits, prompt_ids, max_new_tokens, eos_id,
                       num_beams=num_beams, length_penalty=length_penalty,
                       banned_ids=banned_ids, device=device)


@torch.no_grad()
def seq2seq_generate(model, source_ids, max_new_tokens, eos_id=None, num_beams=1, length_penalty=1.0,
                     decoder_start_id=None, banned_ids=None, device=DEVICE):
    """
    model: EncoderDecoderTransformer (raw or PEFT-wrapped). Uses model.encode() once and
           model(attention_mask=..., decoder_input_ids=..., encoder_hidden_states=...)["logits"].
    """
    eos_id = EOS_ID if eos_id is None else eos_id
    decoder_start_id = ENCDEC_CONFIG.decoder_start_token_id if decoder_start_id is None else decoder_start_id
    banned_ids = (PAD_ID, BOS_ID) if banned_ids is None else banned_ids
    max_new_tokens = min(max_new_tokens, ENCDEC_CONFIG.max_target_length - 1)

    src = torch.tensor([source_ids[:ENCDEC_CONFIG.max_input_length]], dtype=torch.long, device=device)
    src_mask = torch.ones_like(src)
    enc = model.encode(src, src_mask)                    # [1, S, D]

    def next_logits(x):
        n = x.size(0)
        # .repeat (real copy): the encoder-decoder attention reshapes the mask with .view(), which an expanded tensor may not allow
        out = model(attention_mask=src_mask.repeat(n, 1), decoder_input_ids=x,
                    encoder_hidden_states=enc.expand(n, -1, -1))
        return out["logits"][:, -1, :]

    return beam_search(next_logits, [decoder_start_id], max_new_tokens, eos_id,
                       num_beams=num_beams, length_penalty=length_penalty,
                       banned_ids=banned_ids, device=device)

## 7. Backbone registry and checkpoint checks

`adapter_targets` are the `nn.Linear` module-name suffixes wrapped by LoRA / LORMA (for the CNN the attention projections
and the feed-forward layers `ffn.0` / `ffn.2`). `.safetensors`, `.bin`/`.pt` files and checkpoint directories are accepted.

In [ ]:
BACKBONES = {}


def register_backbone(
    name,
    builder,
    hidden_dim,
    checkpoint_path=None,
    adapter_targets=None,
    ia3_feedforward=None,
):
    BACKBONES[name] = {
        "builder": builder,
        "hidden_dim": hidden_dim,
        "checkpoint_path": checkpoint_path,
        # nn.Linear module-name suffixes for LoRA / rsLoRA / IA3 / LORMA
        "adapter_targets": adapter_targets or [],
        # subset of adapter_targets treated as feed-forward by IA3
        "ia3_feedforward": ia3_feedforward or [],
    }


def _load_state(path):
    from pathlib import Path

    path = Path(path)
    if path.is_dir():
        safe_path = path / "model.safetensors"
        bin_path = path / "pytorch_model.bin"
        if safe_path.is_file():
            path = safe_path
        elif bin_path.is_file():
            path = bin_path
        else:
            raise FileNotFoundError(
                f"No model.safetensors or pytorch_model.bin in {path}"
            )

    if str(path).endswith(".safetensors"):
        from safetensors.torch import load_file
        state = load_file(str(path), device="cpu")
    else:
        try:
            state = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            state = torch.load(path, map_location="cpu")

    if isinstance(state, dict):
        for wrapper_key in ("model_state_dict", "state_dict"):
            if wrapper_key in state and isinstance(state[wrapper_key], dict):
                state = state[wrapper_key]
                break
    return state

def load_backbone(name, checkpoint_path=None, device="cpu", strict=True):
    if name not in BACKBONES:
        raise ValueError(f"Unknown backbone: {name}. Available: {list(BACKBONES)}")

    info = BACKBONES[name]
    model = info["builder"]()

    ckpt = checkpoint_path if checkpoint_path is not None else info["checkpoint_path"]
    if ckpt is None:
        raise ValueError(f"No checkpoint supplied for backbone '{name}'")

    result = model.load_state_dict(_load_state(ckpt), strict=strict)

    print("Backbone:", name, "| checkpoint:", ckpt)
    print("Missing keys   :", result.missing_keys)
    print("Unexpected keys:", result.unexpected_keys)

    return model.to(device)

In [ ]:
register_backbone(
    name="decoder",
    builder=lambda: DecoderOnlyTransformer(decoder_config),
    hidden_dim=decoder_config.n_embd,
    checkpoint_path=DECODER_CHECKPOINT,
    adapter_targets=["qkv", "out_proj", "fc_in", "fc_out"],
)

register_backbone(
    name="ssm",
    builder=build_ssm_backbone,
    hidden_dim=SSM_DIM,
    checkpoint_path=SSM_CHECKPOINT,
    adapter_targets=["in_proj", "out_proj"],
)

register_backbone(
    name="encdec",
    builder=build_encdec_backbone,
    hidden_dim=ENCDEC_CONFIG.d_model,
    checkpoint_path=ENCDEC_CHECKPOINT,
    # wi_0 / wi_1 exist only for gated feed-forward configs; unmatched suffixes are ignored
    adapter_targets=["q", "k", "v", "o", "wi", "wi_0", "wi_1", "wo"],
)

register_backbone(
    name="cnn",
    builder=build_cnn_backbone,
    hidden_dim=CNN_DIM,
    checkpoint_path=CNN_CHECKPOINT,
    adapter_targets=["q_proj", "k_proj", "v_proj", "out_proj", "ffn.0", "ffn.2"],
)

assert set(BACKBONES) == {"decoder", "ssm", "encdec", "cnn"}, list(BACKBONES)
print("Registered backbones:", list(BACKBONES))

In [ ]:
# Checkpoint presence (strict loading, encode() and LM logits are tested in section 13).
paths_to_check = {name: info["checkpoint_path"] for name, info in BACKBONES.items()}
paths_to_check["kd_teacher"] = KD_TEACHER_CHECKPOINT
for name, path in paths_to_check.items():
    status = "found" if Path(path).exists() else "MISSING"
    print(f"{name:10s} {status:8s} {path}")

## 8. Phase-6 model wrapper

Causal backbones: `prompt + summary → backbone forward → LM logits [B, T, V]`, loss only on summary positions; generation = `causal_generate()` from the prompt.
Encoder-decoder backbone (`encdec`): `document → encoder`, `summary → decoder` (teacher forcing), generation = `seq2seq_generate()`.

The summarizer reuses the Phase-1 **LM head** (inside the backbone), so by default there is no new task head: under PEFT/LORMA only the adapters train.

In [ ]:
class Phase6SummModel(nn.Module):
    """
    forward(input_ids, attention_mask, labels=None, decoder_input_ids=None, decoder_attention_mask=None, **unused)

    returns {
        "loss"       : scalar or None — token-level CE over summary positions only,
        "logits"     : [B, T, V],
        "target_mask": [B, T] bool — True where logits[:, t] predicts a summary token
                       (causal: target_mask[:, t] = labels[:, t + 1] != -100, last column False;
                        enc-dec: target_mask = labels != -100)
    }
    target_mask lets the KD loss (Section 11) compare teacher/student on exactly the summary tokens.
    """

    def __init__(self, backbone, hidden_dim, encoder_decoder=False,
                 label_smoothing=0.0, memory_slots=64, max_chunks=256, **unused):
        super().__init__()
        self.backbone = backbone
        self.hidden_dim = hidden_dim
        self.encoder_decoder = encoder_decoder
        self.label_smoothing = label_smoothing


        self.memory_slots = memory_slots
        self.max_chunks = max_chunks
        assert self.memory_slots >= 1

        if self.encoder_decoder and not STANDARD_ENCDEC:
            # Preserve the order of chunks in the original document.
            self.chunk_position = nn.Embedding(max_chunks, hidden_dim)
            nn.init.normal_(self.chunk_position.weight, std=0.02)

    def head_module_names(self):
        """Task-specific submodules outside the backbone that must stay trainable (none by default)."""
        if self.encoder_decoder and not STANDARD_ENCDEC:
            return ["chunk_position"]
        return []

    def encode_document(self, input_ids, attention_mask, chunk_mask=None):
        """
        Hierarchical document encoding.

        input_ids      : [B, C, S]
        attention_mask : [B, C, S]
        chunk_mask     : [B, C]

        Returns:
            memory      : [B, M, D]
            memory_mask : [B, M]

        Each document chunk is encoded using the pretrained Phase-1 encoder.
        Up to memory_slots actual token representations are selected from each chunk.
        The representations are concatenated into document-level memory.
        """

        assert self.encoder_decoder
        assert input_ids.ndim == 3

        B, C, S = input_ids.shape
        assert attention_mask.shape == input_ids.shape
        assert S <= self.backbone.config.max_input_length
        assert C <= self.max_chunks

        if chunk_mask is None:
            chunk_mask = attention_mask.any(dim=-1)

        assert chunk_mask.shape == (B, C)

        all_memory = []
        all_masks = []

        for b in range(B):
            doc_memory = []

            for c in range(C):
                if not bool(chunk_mask[b, c].item()):
                    continue

                ids = input_ids[b, c]
                mask = attention_mask[b, c]

                # Remove padding before sending a chunk to the encoder.
                n = int(mask.sum().item())
                assert n > 0

                ids = ids[:n].unsqueeze(0)
                mask = mask[:n].unsqueeze(0)

                # Existing pretrained Phase-1 encoder.
                # [1, S, D] -> [S, D]
                from torch.utils.checkpoint import checkpoint

                if self.training and torch.is_grad_enabled():
                    h = checkpoint(
                        lambda x, m: self.backbone.encode(x, m),
                        ids,
                        mask,
                        use_reentrant=False
                    ).squeeze(0)
                else:
                    h = self.backbone.encode(ids, mask).squeeze(0)

                # Select actual pretrained token representations, not learned averages.
                # [S, D] -> [min(memory_slots, S), D]
                k = min(self.memory_slots, h.size(0))
                idx = torch.linspace(
                    0, h.size(0) - 1, steps=k, device=h.device
                ).long()
                pooled = h.index_select(0, idx)

                # Add the original chunk's document position.
                pos = self.chunk_position(
                    torch.tensor(c, device=h.device)
                )

                pooled = pooled + pos.unsqueeze(0)
                doc_memory.append(pooled)

            assert doc_memory, "Document contains no valid chunks"

            # [sum(min(memory_slots, chunk_length)), hidden_dim]
            doc_memory = torch.cat(doc_memory, dim=0)

            all_memory.append(doc_memory)
            all_masks.append(
                torch.ones(
                    doc_memory.shape[0],
                    dtype=torch.long,
                    device=doc_memory.device
                )
            )

        # Pad document-level memory for batches containing
        # different numbers of chunks.
        memory = nn.utils.rnn.pad_sequence(
            all_memory, batch_first=True, padding_value=0.0
        )

        memory_mask = nn.utils.rnn.pad_sequence(
            all_masks, batch_first=True, padding_value=0
        )

        return memory, memory_mask

    def forward(self, input_ids, attention_mask, labels=None,
                decoder_input_ids=None, decoder_attention_mask=None,
                chunk_mask=None, **unused):

        if self.encoder_decoder:
            if STANDARD_ENCDEC:
                # Original Phase-1 encoder-decoder interface, no compressed memory.
                out = self.backbone(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    decoder_input_ids=decoder_input_ids,
                    decoder_attention_mask=decoder_attention_mask
                )
            else:
                # Hierarchical document encoding.
                memory, memory_mask = self.encode_document(
                    input_ids, attention_mask, chunk_mask
                )

                # Reuse the original Phase-1 decoder, LM head,
                # shift_right() and tied-embedding scaling.
                out = self.backbone(
                    encoder_hidden_states=memory,
                    attention_mask=memory_mask,
                    labels=labels,
                    decoder_input_ids=decoder_input_ids,
                    decoder_attention_mask=decoder_attention_mask
                )

            logits = out["logits"]
            loss = None
            target_mask = None

            if labels is not None:
                assert logits.shape[:2] == labels.shape

                target_mask = labels != -100

                loss = F.cross_entropy(
                    logits.float().reshape(-1, logits.shape[-1]),
                    labels.reshape(-1),
                    ignore_index=-100,
                    label_smoothing=self.label_smoothing
                )

            return {
                "loss": loss,
                "logits": logits,
                "target_mask": target_mask
            }

        # Standard causal backbone: Decoder / CNN / SSM.
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = out["logits"]
        loss = None
        target_mask = None

        if labels is not None:
            assert logits.shape[:2] == labels.shape

            # A causal logit at position t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.float().reshape(-1, logits.shape[-1]),
                shift_labels.reshape(-1),
                ignore_index=-100,
                label_smoothing=self.label_smoothing
            )

            target_mask = torch.zeros_like(labels, dtype=torch.bool)
            target_mask[:, :-1] = labels[:, 1:] != -100

        return {
            "loss": loss,
            "logits": logits,
            "target_mask": target_mask
        }

    @torch.no_grad()
    def generate_summaries(self, documents, cfg, tokenizer, device=DEVICE):
        """
        documents : List[str] raw source documents
        returns   : List[str] decoded summaries (same order), using
                    cfg.max_generation_length / cfg.num_beams / cfg.length_penalty
        """

        was_training = self.training
        self.eval()

        summaries = []

        try:
            for document in documents:
                if self.encoder_decoder and STANDARD_ENCDEC:
                    # Standard pretrained encoder-decoder generation.
                    src = tokenizer.encode(document)[:cfg.max_source_tokens] + [EOS_ID]
                    ids = seq2seq_generate(
                        self.backbone,
                        source_ids=src,
                        max_new_tokens=min(cfg.max_generation_length, self.backbone.config.max_target_length - 1),
                        num_beams=cfg.num_beams,
                        length_penalty=cfg.length_penalty,
                        device=device
                    )

                elif self.encoder_decoder:
                    # Tokenize the entire document and preserve all chunks.
                    source_ids = tokenizer.encode(document)

                    chunks = chunk_document(
                        source_ids,
                        chunk_size=cfg.max_source_tokens,
                        overlap=getattr(cfg, "chunk_overlap", CHUNK_OVERLAP)
                    )

                    chunks = [c + [EOS_ID] for c in chunks]

                    C = len(chunks)
                    S = max(map(len, chunks))

                    assert S <= self.backbone.config.max_input_length

                    src = torch.full(
                        (1, C, S),
                        PAD_ID,
                        dtype=torch.long,
                        device=device
                    )

                    mask = torch.zeros_like(src)
                    cmask = torch.ones(
                        (1, C), dtype=torch.long, device=device
                    )

                    for j, chunk in enumerate(chunks):
                        n = len(chunk)
                        src[0, j, :n] = torch.tensor(
                            chunk, dtype=torch.long, device=device
                        )
                        mask[0, j, :n] = 1

                    # Compute document memory ONCE.
                    memory, memory_mask = self.encode_document(
                        src, mask, cmask
                    )

                    max_new = min(
                        cfg.max_generation_length,
                        self.backbone.config.max_target_length - 1
                    )

                    def next_logits(x):
                        n = x.size(0)

                        out = self.backbone(
                            encoder_hidden_states=memory.expand(n, -1, -1),
                            attention_mask=memory_mask.repeat(n, 1),
                            decoder_input_ids=x
                        )

                        return out["logits"][:, -1, :]

                    ids = beam_search(
                        next_logits,
                        start_tokens=[
                            self.backbone.config.decoder_start_token_id
                        ],
                        max_new_tokens=max_new,
                        eos_id=EOS_ID,
                        num_beams=cfg.num_beams,
                        length_penalty=cfg.length_penalty,
                        banned_ids=(PAD_ID, BOS_ID),
                        device=device
                    )

                else:
                    # Standard causal generation for the other three backbones.
                    prompt = build_prompt_ids(
                        document, tokenizer, cfg.max_source_tokens
                    )

                    ids = causal_generate(
                        self.backbone,
                        prompt,
                        cfg.max_generation_length,
                        num_beams=cfg.num_beams,
                        length_penalty=cfg.length_penalty,
                        context_length=BACKBONE_MAX_TOKENS[cfg.backbone_name],
                        device=device
                    )

                summaries.append(tokenizer.decode(ids).strip())

        finally:
            self.train(was_training)

        return summaries


def build_phase6_model(backbone_name, device, checkpoint_path=None, backbone=None, **model_kwargs):
    """Generic builder: Phase-1 backbone (loaded from registry unless given) + Phase-6 wrapper."""
    if backbone is None:
        backbone = load_backbone(backbone_name, checkpoint_path=checkpoint_path, device="cpu")
    model = Phase6SummModel(
        backbone=backbone,
        hidden_dim=BACKBONES[backbone_name]["hidden_dim"],
        encoder_decoder=(backbone_name == "encdec"),
        **model_kwargs,
    )
    return model.to(device)

## 9. Long-document handling

- **Causal backbones:** the prompt keeps the first `MAX_SOURCE_TOKENS` document tokens (`build_prompt_ids`); training sequences are
  `1 + MAX_SOURCE_TOKENS + |" Summary:"| + MAX_TARGET_TOKENS ≤ 2048`. During generation `causal_generate` slides the window if prompt + output exceed the context.
- **Encoder-decoder, standard:** the encoder sees the first `MAX_SOURCE_TOKENS` tokens + EOS (≤ 1024).
- **Encoder-decoder, hierarchical:** every overlapping chunk is encoded; `memory_slots` evenly spaced token states per chunk plus a learned chunk-position embedding form the decoder memory.
- **Targets:** summaries longer than `MAX_TARGET_TOKENS − 1` are truncated without EOS (checked in section 5).
- **Generation length:** `max_generation_length` new tokens (default 384), greedy (`num_beams = 1`).

## 10. Adaptation methods

Two adaptation methods are registered here (`lora`, `lorma`); knowledge distillation is in section 15.

| key | what is trained | update applied to each targeted `nn.Linear` |
|-----|-----------------|---------------------------------------------|
| `lora` | `A ∈ ℝ^(r×d_in)`, `B ∈ ℝ^(d_out×r)` | `y = W₀x + (α/r)·B A x` (additive) |
| `lorma` | `A ∈ ℝ^(r×d_out)`, `B ∈ ℝ^(d_out×r)` (`B = 0` at init ⇒ identity) | `y = W₀x + b + (α/r)·B A (W₀x)` = `(I + (α/r)·BA)·W₀x + b` (**multiplicative**) |

All methods are applied **to the backbone only**; the Phase-6 head (`model.head_module_names()`) is always trainable.
For PEFT methods the backbone is placed in `PeftBackboneShim` and wrapped by `get_peft_model`; `.encode()` is forwarded to the Phase-1 model and `forward()` reaches the Phase-1 LM head (needed for generation).

> Checkpoints saved after adaptation contain adapter keys. To reload: **build the model → apply the same adaptation → `load_state_dict`** (use `build_adapted_model`).

In [ ]:
class LowRankMulLinear(nn.Module):
    """LORMA: frozen base nn.Linear + trainable low-rank update (from Phase 2)."""

    def __init__(self, base_layer, rank=16, alpha=32.0):
        super().__init__()

        if not isinstance(base_layer, nn.Linear):
            raise TypeError("base_layer must be nn.Linear")

        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        in_dim = base_layer.in_features
        out_dim = base_layer.out_features
        device = base_layer.weight.device
        dtype = base_layer.weight.dtype
        #LORA
        # self.A = nn.Parameter(torch.empty(rank, in_dim, device=device, dtype=dtype))
        # self.B = nn.Parameter(torch.zeros(out_dim, rank, device=device, dtype=dtype))
        # nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        #LORMA
        
        self.A = nn.Parameter(
            torch.empty(rank, out_dim, device=device, dtype=dtype)
        )
        
        self.B = nn.Parameter(
            torch.zeros(out_dim, rank, device=device, dtype=dtype)
        )
        
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        #LORA
    # def forward(self, x):
    #     return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

        #LORMA
    def forward(self, x):
        y = self.base(x)
    
        # Apply the multiplicative transformation to W0*x,
        # excluding the frozen bias from the transformation.
        pre = y if self.base.bias is None else y - self.base.bias
    
        update = F.linear(
            F.linear(pre, self.A),
            self.B
        )
    
        return y + self.scale * update


def inject_low_rank(module, target_suffixes, rank, alpha):
    """Replace every nn.Linear whose name ends with a target suffix by LowRankMulLinear. Generic across backbones."""
    to_replace = [
        name for name, m in module.named_modules()
        if isinstance(m, nn.Linear)
        and any(name == t or name.endswith("." + t) for t in target_suffixes)
    ]
    for name in to_replace:
        parent_name, _, child = name.rpartition(".")
        parent = module.get_submodule(parent_name) if parent_name else module
        setattr(parent, child, LowRankMulLinear(getattr(parent, child), rank=rank, alpha=alpha))
    return to_replace

In [ ]:
ADAPTATION_METHODS = {}


def register_adaptation(name):
    def deco(fn):
        ADAPTATION_METHODS[name] = fn
        return fn
    return deco


def _freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False


def _unfreeze_head(model):
    for name in model.head_module_names():
        for p in getattr(model, name).parameters():
            p.requires_grad = True


def _targets(cfg):
    t = BACKBONES[cfg.backbone_name]["adapter_targets"]
    if not t:
        raise ValueError(f"No adapter_targets registered for backbone '{cfg.backbone_name}'")
    return t


class PeftBackboneShim(nn.Module):
    """
    Thin wrapper so PEFT never inspects the Phase-1 model's own `.config`
    (a plain dataclass, not a HF PretrainedConfig). Forwards encode() and forward() (LM logits).
    Target-module suffix matching (e.g. "qkv") still works on "model.blocks.0.attn.qkv".
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    @property
    def config(self):
        return self.model.config
        
    def encode(self, input_ids, attention_mask=None):
        return self.model.encode(input_ids, attention_mask)

    def forward(self, *args, **kwargs):
        # Phase 6: forward() must reach the Phase-1 LM head (logits) for training and generation
        return self.model(*args, **kwargs)


def _wrap_with_peft(model, peft_cfg):
    model.backbone = get_peft_model(PeftBackboneShim(model.backbone), peft_cfg)
    model.backbone.print_trainable_parameters()
    return model


# ------------------------------------------------------------
# 1. LoRA (PEFT)
# ------------------------------------------------------------
@register_adaptation("lora")
def apply_lora(model, cfg):
    _freeze_backbone(model)
    peft_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=_targets(cfg),
        bias="none",
        use_rslora=False,
    )
    _wrap_with_peft(model, peft_cfg)
    _unfreeze_head(model)
    return model




# ------------------------------------------------------------
# 2. LORMA (custom LowRankLinear injection)
# ------------------------------------------------------------
@register_adaptation("lorma")
def apply_lorma(model, cfg):
    _freeze_backbone(model)
    replaced = inject_low_rank(
        model.backbone,
        target_suffixes=_targets(cfg),
        rank=cfg.lorma_rank,
        alpha=cfg.lorma_alpha,
    )
    if not replaced:
        raise RuntimeError(
            f"LORMA: No matching linear layers found for {_targets(cfg)}"
        )
    print(f"LORMA: wrapped {len(replaced)} Linear layers")
    _unfreeze_head(model)
    return model


def apply_adaptation(model, cfg):
    if cfg.adaptation_method not in ADAPTATION_METHODS:
        raise ValueError(
            f"Unknown adaptation '{cfg.adaptation_method}'. "
            f"Available: {list(ADAPTATION_METHODS)}"
        )
    model = ADAPTATION_METHODS[cfg.adaptation_method](model, cfg)
    report_trainable(model)
    return model


def report_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params    : {total:,}")
    print(f"Trainable params: {trainable:,} ({100 * trainable / max(total, 1):.3f}%)")
    return total, trainable


assert set(ADAPTATION_METHODS) == {"lora",  "lorma"}, ADAPTATION_METHODS.keys()
print("Adaptation methods:", list(ADAPTATION_METHODS))

## 11. Experiment configuration

`exp_cfg` is trained by section 14 when `RUN_TRAINING = True`. The active configuration is the best recorded run
(`dec_lorma`: decoder + LORMA r16). For an encoder-decoder run set `backbone_name="encdec"` and `ENCODER_DECODER = True`
in section 2 (`check_length_budget` refuses a mismatch).

In [ ]:
@dataclass
class ExperimentConfig:
    backbone_name: str
    adaptation_method: str          # "lora" | "lorma"

    learning_rate: float            # adapter / backbone-side LR
    batch_size: int
    grad_accum: int
    epochs: int
    # Hierarchical encoder-decoder
    memory_slots: int = 64
    max_chunks: int = 256
    chunk_overlap: int = CHUNK_OVERLAP
    
    head_learning_rate: float = 3e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool = True
    dropout: float = 0.1
    label_smoothing: float = 0.0

    # Source / target lengths used for training and prompting
    max_source_tokens: int = MAX_SOURCE_TOKENS
    max_target_tokens: int = MAX_TARGET_TOKENS

    # Generation (validation + test)
    max_generation_length: int = 384     # max NEW tokens per summary
    num_beams: int = 1                   # 1 = greedy
    length_penalty: float = 1.0          # beam score = logprob / len**length_penalty

    # Validation-time generation (BERTScore is the selection metric; generation is slow)
    val_generate: bool = True
    val_max_docs: int = 50               # number of val docs summarised per epoch

    # PEFT (LoRA / rsLoRA)
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # LORMA
    lorma_rank: int = 16
    lorma_alpha: float = 32.0

    seed: int = SEED
    run_name: Optional[str] = None

    @property
    def run_dir(self):
        name = self.run_name or f"phase6_{self.backbone_name}_{self.adaptation_method}"
        return WORK_DIR / name


def save_experiment_config(cfg, run_dir=None):
    run_dir = Path(run_dir or cfg.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)



# Best recorded run (Codabench 0.7876): decoder + LORMA.
exp_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="lorma",
    learning_rate=5e-4,
    batch_size=BATCH_SIZE,
    grad_accum=16,
    epochs=4,

    # LORMA
    lorma_rank=16,
    lorma_alpha=32.0,


    # Generation
    max_generation_length=384,
    num_beams=1,
    length_penalty=1.0,

    # Validation
    val_generate=True,
    val_max_docs=50,

    memory_slots=64,  
    max_chunks=256,
    chunk_overlap=128,
    run_name="dec_lorma"
)
# Decoder + LoRA: same settings with adaptation_method="lora", run_name="dec_lora_standard"


assert exp_cfg.backbone_name in BACKBONES, exp_cfg.backbone_name
assert exp_cfg.adaptation_method in ADAPTATION_METHODS, exp_cfg.adaptation_method
print(asdict(exp_cfg))

In [ ]:
def build_adapted_model(cfg, device=DEVICE, backbone=None, **extra_model_kwargs):
    """Phase-1 backbone + Phase-6 head + chosen adaptation. Use this both for training and for reloading checkpoints."""
    set_seed(cfg.seed)

    # Hierarchical encoder-decoder configuration must match during training and checkpoint reload.
    model_kwargs = {
        "memory_slots": cfg.memory_slots,
        "max_chunks": cfg.max_chunks,
    }
    model_kwargs.update(extra_model_kwargs)

    model = build_phase6_model(
        backbone_name=cfg.backbone_name,
        device="cpu",
        backbone=backbone,
        label_smoothing=cfg.label_smoothing,
        **model_kwargs,
    )

    model = apply_adaptation(model, cfg)
    return model.to(device)


def build_optimizer(model, cfg):
    head_names = tuple(f"{n}." for n in model.head_module_names())
    head_params, adapter_params = [], []

    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith(head_names) else adapter_params).append(p)

    groups = []
    if adapter_params:
        groups.append({"params": adapter_params, "lr": cfg.learning_rate})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.head_learning_rate})

    assert groups, "No trainable parameters found"

    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)


def build_scheduler(optimizer, cfg, steps_per_epoch):
    total_steps = max(math.ceil(steps_per_epoch / cfg.grad_accum) * cfg.epochs, 1)

    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[g["lr"] for g in optimizer.param_groups],
        total_steps=total_steps,
        pct_start=0.1,
        anneal_strategy="cos",
    )



def load_trained_model(cfg, checkpoint_path=None, backbone=None):
    """Rebuild backbone + wrapper + adaptation for `cfg`, then load a trained Phase-6 checkpoint strictly."""
    path = Path(checkpoint_path or (cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"Trained checkpoint not found: {path}")
    model = build_adapted_model(cfg, backbone=backbone)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
    model.eval()
    print("Loaded trained model:", path)
    return model

In [ ]:
def check_length_budget(cfg):
    """Check that training sequences and generation fit the chosen Phase-1 backbone."""
    is_encdec = cfg.backbone_name == "encdec"
    assert ENCODER_DECODER == is_encdec, "set ENCODER_DECODER in Section 3 to match exp_cfg.backbone_name"

    if is_encdec:
        # Hierarchical encoder-decoder: source length is PER CHUNK.
        src_len, tgt_len = cfg.max_source_tokens + 1, cfg.max_target_tokens

        print(f"encdec chunk <= {src_len} (max {ENCDEC_CONFIG.max_input_length}) | "
              f"target <= {tgt_len} / generation <= {cfg.max_generation_length + 1} "
              f"(max {ENCDEC_CONFIG.max_target_length})")

        assert src_len <= ENCDEC_CONFIG.max_input_length, "reduce max_source_tokens (encdec input limit 1024)"
        assert tgt_len <= ENCDEC_CONFIG.max_target_length, "reduce max_target_tokens"
        assert cfg.max_generation_length + 1 <= ENCDEC_CONFIG.max_target_length, "reduce max_generation_length"

        if STANDARD_ENCDEC:
            assert train_ds.encoder_decoder
            assert all(len(x["source_ids"]) <= ENCDEC_CONFIG.max_input_length for x in train_ds.items)
            assert max(len(x["target_ids"]) for x in train_ds.items) <= cfg.max_target_tokens
            print("Standard encoder-decoder length checks passed.")
            return

        # Hierarchical document chunking checks.
        assert 0 <= cfg.chunk_overlap < cfg.max_source_tokens
        assert cfg.memory_slots > 0
        assert cfg.max_chunks > 0

        # Ensure Dataset and experiment configuration agree.
        assert train_ds.encoder_decoder
        assert train_ds.items
        assert max(len(x["target_ids"]) for x in train_ds.items) <= cfg.max_target_tokens

        # Check the actual number of chunks in the available splits.
        max_train_chunks = max(len(x["chunks"]) for x in train_ds.items)
        max_val_chunks = max(len(x["chunks"]) for x in val_ds.items)

        print(f"Max chunks: train={max_train_chunks}, val={max_val_chunks}, "
              f"model limit={cfg.max_chunks}")

        assert max(max_train_chunks, max_val_chunks) <= cfg.max_chunks, \
            "Some documents exceed max_chunks"

        return

    context_length = BACKBONE_MAX_TOKENS[cfg.backbone_name]
    sep = len(tokenizer.encode(SUMMARY_PROMPT))
    train_len = 1 + cfg.max_source_tokens + sep + cfg.max_target_tokens
    gen_len = 1 + cfg.max_source_tokens + sep + cfg.max_generation_length + 1

    print(f"train sequence <= {train_len} | generation window <= {gen_len} | context {context_length}")

    assert train_len <= context_length, "reduce max_source_tokens / max_target_tokens"

    if gen_len > context_length:
        print("[warn] generation exceeds the window: causal_generate will slide (drop earliest prompt tokens)")


check_length_budget(exp_cfg)

## 12. Training, evaluation and submission utilities

Definitions only — nothing in this section trains a model.

`train_one_epoch()` is the generic teacher-forced loop. `run_validation()` computes the teacher-forced validation loss and
**auto-regressively generates** summaries for up to `cfg.val_max_docs` validation documents.

In [ ]:
def split_batch(batch, device):
    tensors, meta = {}, {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            tensors[k] = v.to(device, non_blocking=True)
        else:
            meta[k] = v
    return tensors, meta


def _amp_enabled(use_amp):
    return use_amp and torch.cuda.is_available()


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    grad_accum=1,
    max_grad_norm=1.0,
    use_amp=True,
    scaler=None,
    scheduler=None,
):
    model.train()
    amp = _amp_enabled(use_amp)

    if scaler is None:
        scaler = make_scaler(amp)

    assert grad_accum >= 1

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    num_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False)):
        tensors, _ = split_batch(batch, device)

        # Handle incomplete accumulation groups at the end of an epoch.
        remaining = num_batches - (step // grad_accum) * grad_accum
        current_accum = min(grad_accum, remaining)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp
        ):
            out = model(**tensors)
            raw_loss = out["loss"]
            loss = raw_loss / current_accum

        if not torch.isfinite(raw_loss):
            raise RuntimeError(
                f"Non-finite training loss at batch {step}"
            )

        scaler.scale(loss).backward()

        update_now = (
            (step + 1) % grad_accum == 0
            or (step + 1) == num_batches
        )

        if update_now:
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    max_grad_norm
                )

            old_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()

            step_succeeded = (
                not amp or scaler.get_scale() >= old_scale
            )

            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None and step_succeeded:
                scheduler.step()

        total_loss += raw_loss.detach().item()

        # Release batch references before the next iteration.
        del out, raw_loss, loss, tensors

    return total_loss / max(num_batches, 1)


@torch.no_grad()
def run_validation(model, loader, val_docs, cfg, device=DEVICE, generate=True):
    """
    Returns:
        loss       : mean teacher-forced loss on the val loader
        hypotheses : List[str] generated summaries (first cfg.val_max_docs val docs) — if generate
        references : List[str] gold summaries aligned with hypotheses
        ids        : List[str]
    """
    model.eval()
    amp = _amp_enabled(cfg.use_amp)

    total_loss, n_loss = 0.0, 0

    for batch in tqdm(loader, desc="val-loss", leave=False):
        tensors, _ = split_batch(batch, device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp
        ):
            out = model(**tensors)

        if out.get("loss") is not None and torch.isfinite(out["loss"]):
            total_loss += out["loss"].item()
            n_loss += 1

        del out, tensors

    result = {
        "loss": total_loss / n_loss if n_loss else None,
        "hypotheses": [],
        "references": [],
        "ids": []
    }

    if generate:

        subset = val_docs[:cfg.val_max_docs]

        documents = [d["document"] for d in subset]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp
        ):
            hyps = model.generate_summaries(
                documents,
                cfg,
                tokenizer,
                device=device
            )

        assert len(hyps) == len(subset), \
            "Number of generated summaries does not match validation documents"

        result["hypotheses"] = hyps
        result["references"] = [d["summary"] for d in subset]
        result["ids"] = [d["id"] for d in subset]

        assert len(set(result["ids"])) == len(result["ids"]), \
            "Duplicate validation IDs"

        print(f"Generated {len(hyps)} validation summaries")

    return result

### 12.1 Metrics

BERTScore F1 (primary), ROUGE-1 F1 and ROUGE-L F1 — all computed on **decoded strings** (generated summary vs reference), averaged over documents.

BERTScore downloads its scoring model on first use (evaluation only).

In [ ]:
BERTSCORE_KWARGS = dict(lang="en")   # match the official evaluator's model_type / rescaling if it is specified


def compute_summ_metrics(hypotheses, references):
    """
    hypotheses, references : List[str], aligned

    returns {"bertscore_f1": float, "rouge1_f1": float, "rougeL_f1": float, "num_docs": int}
    """

    assert len(hypotheses) == len(references) > 0

    hypotheses = [str(h or "").strip() for h in hypotheses]
    references = [str(r or "").strip() for r in references]

    valid = [
        i for i, (h, r) in enumerate(zip(hypotheses, references))
        if h and r
    ]

    bert_f1 = [0.0] * len(hypotheses)

    if valid:
        hyps = [hypotheses[i] for i in valid]
        refs = [references[i] for i in valid]

        _, _, F = bert_score(
            hyps,
            refs,
            **BERTSCORE_KWARGS
        )

        for i, f in zip(valid, F.tolist()):
            bert_f1[i] = float(f)

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"],
        use_stemmer=True
    )

    rouge1, rougeL = [], []

    for hyp, ref in zip(hypotheses, references):
        if not hyp or not ref:
            rouge1.append(0.0)
            rougeL.append(0.0)
            continue

        scores = scorer.score(ref, hyp)

        rouge1.append(scores["rouge1"].fmeasure)
        rougeL.append(scores["rougeL"].fmeasure)

    n = len(hypotheses)

    return {
        "bertscore_f1": float(sum(bert_f1) / n),
        "rouge1_f1": float(sum(rouge1) / n),
        "rougeL_f1": float(sum(rougeL) / n),
        "num_docs": n
    }

### 12.2 Training loop

In [ ]:
def fit(cfg, train_loader, val_loader):
    run_dir = cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(cfg)

    model = build_adapted_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler = make_scaler(_amp_enabled(cfg.use_amp))

    best_score, history = -float("inf"), []

    for epoch in range(cfg.epochs):
        t0 = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, DEVICE,
            grad_accum=cfg.grad_accum, max_grad_norm=cfg.max_grad_norm,
            use_amp=cfg.use_amp, scaler=scaler, scheduler=scheduler,
        )
        val_out = run_validation(model, val_loader, val_docs, cfg, generate=cfg.val_generate)

        row = {"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_out["loss"]}
        if cfg.val_generate:
            metrics = compute_summ_metrics(val_out["hypotheses"], val_out["references"])   # Section 10
            row.update(metrics)
            score = metrics["bertscore_f1"]
        else:
            metrics = {}
            score = -val_out["loss"]
        row["time_s"] = round(time.time() - t0, 1)
        history.append(row)
        print(row)

        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), run_dir / "best_model.pt")
            with open(run_dir / "metrics.json", "w") as f:
                json.dump(row, f, indent=2)
            if cfg.val_generate:
                with open(run_dir / "val_samples.json", "w", encoding="utf-8") as f:
                    json.dump([{"id": i, "hyp": h, "ref": r} for i, h, r in
                               zip(val_out["ids"], val_out["hypotheses"], val_out["references"])][:20],
                              f, ensure_ascii=False, indent=2)
            print(f"  ✓ new best ({'BERTScore F1' if cfg.val_generate else '-val_loss'}) {best_score:.4f}")

    with open(run_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    # reload the best checkpoint (as in Phase 3) so the returned model is the selected one
    model.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=DEVICE, weights_only=True))
    print(f"Training complete. Best: {best_score:.4f}")
    return model, best_score, history

In [ ]:
def tiny_overfit_test(cfg, n_items=2, steps=40):
    model = build_adapted_model(cfg)
    tiny_loader = DataLoader(
        Subset(train_ds, range(min(n_items, len(train_ds)))),
        batch_size=1, shuffle=True, collate_fn=summ_collator,
    )

    opt = build_optimizer(model, cfg)

    print("Actual learning rates:", [g["lr"] for g in opt.param_groups])

    for i in range(steps):
        loss = train_one_epoch(
            model, tiny_loader, opt, DEVICE,
            grad_accum=1,
            max_grad_norm=1.0,
            use_amp=cfg.use_amp
        )

        if i % 5 == 0 or i == steps - 1:
            print(f"step {i:3d} loss {loss:.4f}")

    hyp = model.generate_summaries(
        [train_docs[0]["document"]],
        replace(cfg, max_generation_length=64),
        tokenizer
    )[0]

    print("REF:", train_docs[0]["summary"][:300])
    print("HYP:", hyp[:300])

    # return model

### 12.3 Test inference and submission helpers

Expected `prediction.json` — a JSON **list**, exactly one generated summary per test document:

```json
[
  {"id": "doc_001", "summary": "Generated text..."},
  {"id": "doc_002", "summary": "Generated text..."}
]
```

Generate from a **freshly loaded** best checkpoint with the decoding settings recorded in `exp_cfg`.

In [ ]:
@torch.no_grad()
def run_inference(model, test_docs, cfg, tokenizer, device=DEVICE, batch_docs=8, save_every=None):
    """
    returns [{"id": doc_id, "summary": str}, ...] — one entry per test document, test-file order
    """

    assert batch_docs >= 1
    assert save_every is None or save_every >= 1

    model.eval()
    predictions = []

    save_path = cfg.run_dir / "inference_partial.json"

    for start in tqdm(range(0, len(test_docs), batch_docs), desc="inference"):
        group = test_docs[start:start + batch_docs]

        summaries = model.generate_summaries(
            [d["document"] for d in group],
            cfg,
            tokenizer,
            device=device
        )

        assert len(summaries) == len(group)

        for doc, summary in zip(group, summaries):
            summary = str(summary or "").strip()

            if not summary:
                text = str(doc["document"]).strip()
                summary = text.split(".")[0].strip()
                if not summary:
                    summary = text[:200].strip()
                if not summary:
                    summary = "No summary available."

            predictions.append({
                "id": doc["id"],
                "summary": summary
            })

        if save_every is not None:
            if len(predictions) % save_every == 0 or len(predictions) == len(test_docs):
                save_path.parent.mkdir(parents=True, exist_ok=True)
                with open(save_path, "w", encoding="utf-8") as f:
                    json.dump(predictions, f, ensure_ascii=False, indent=2)

    return predictions


def validate_submission_format(predictions, test_docs, max_summary_tokens=None):
    """
    Must raise on any violation:
      1. predictions is a list of dicts with exactly the keys {"id", "summary"}
      2. exactly one summary per test document: same id set, no duplicates, no extra ids
      3. every "summary" is a non-empty str
      4. (optional) len(tokenizer.encode(summary)) <= max_summary_tokens  (generation limit respected)
    """

    assert isinstance(predictions, list)
    assert isinstance(test_docs, list)
    assert len(predictions) == len(test_docs)

    expected_ids = [d["id"] for d in test_docs]
    predicted_ids = []

    assert len(set(expected_ids)) == len(expected_ids), "Duplicate test IDs"

    for i, p in enumerate(predictions):
        assert isinstance(p, dict), f"Prediction {i} must be a dictionary"
        assert set(p.keys()) == {"id", "summary"}, f"Invalid keys in prediction {i}"

        assert isinstance(p["id"], type(expected_ids[i])), f"Invalid ID type at {i}"
        assert isinstance(p["summary"], str), f"Summary {i} must be a string"
        assert p["summary"].strip(), f"Empty summary at {i}"

        predicted_ids.append(p["id"])

        if max_summary_tokens is not None:
            n = len(tokenizer.encode(p["summary"]))
            assert n <= max_summary_tokens, (
                f"Summary {i} exceeds token limit: {n} > {max_summary_tokens}"
            )

    assert len(set(predicted_ids)) == len(predicted_ids), "Duplicate prediction IDs"
    assert set(predicted_ids) == set(expected_ids), "Missing or extra prediction IDs"
    assert predicted_ids == expected_ids, "Predictions are not in test-file order"

    print(f"Submission format valid: {len(predictions)} summaries")
    return True

In [ ]:
def save_submission(predictions, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pred_path = output_dir / "prediction.json"
    zip_path = output_dir / "submission.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

    print("Saved:", pred_path)
    print("Saved:", zip_path)
    return pred_path, zip_path


def verify_zip_root(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"zip must contain only prediction.json at root, got {names}"
        obj = json.loads(z.read("prediction.json").decode("utf-8"))
    print("Zip root OK:", names, "| entries:", len(obj))
    return obj

In [ ]:
def enforce_generation_limit(predictions, limit):
    """Trim each summary so that re-tokenising it gives at most `limit` tokens (never empties a summary)."""
    for p in predictions:
        ids = tokenizer.encode(p["summary"].strip())
        while len(ids) > limit:
            ids = ids[:-1]
        trimmed = tokenizer.decode(ids).strip()
        if trimmed:
            p["summary"] = trimmed
        assert p["summary"], f"Empty summary: {p['id']}"
    return predictions

### 12.4 Knowledge-distillation utilities

Definitions used by section 15 (and the KD-loss preflight check).

In [ ]:
class DistillationExperiment:
    """
    Generic teacher -> student distillation (same structure as Phase 3, task-specific KD term plugged in).
      kd_loss_fn(student_out, teacher_out, tensors, temperature) -> scalar soft-target loss
    Total loss = (1 - alpha) * gold_loss(student) + alpha * kd_loss
    """

    def __init__(self, teacher, student, kd_loss_fn, temperature=2.0, alpha=0.5, teacher_drop_keys=()):
        self.teacher = teacher
        self.student = student
        self.kd_loss_fn = kd_loss_fn
        self.temperature = temperature
        self.alpha = alpha
        self.teacher_drop_keys = set(teacher_drop_keys)   # e.g. gold labels: the teacher needs no gold loss

        self.teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad = False

    def compute_loss(self, tensors):
        with torch.no_grad():
            teacher_out = self.teacher(**{k: v for k, v in tensors.items() if k not in self.teacher_drop_keys})

        student_out = self.student(**tensors)
        gold_loss = student_out["loss"]
        kd_loss = self.kd_loss_fn(student_out, teacher_out, tensors, self.temperature)

        loss = (1.0 - self.alpha) * gold_loss + self.alpha * kd_loss
        return {"loss": loss, "gold_loss": gold_loss, "kd_loss": kd_loss}


def train_kd_epoch(
    kd_exp,
    loader,
    optimizer,
    device,
    grad_accum=1,
    max_grad_norm=1.0,
    use_amp=True,
    scaler=None,
    scheduler=None,
):
    kd_exp.student.train()
    kd_exp.teacher.eval()
    amp = _amp_enabled(use_amp)
    if scaler is None:
        scaler = make_scaler(amp)

    optimizer.zero_grad(set_to_none=True)
    tot = tot_gold = tot_kd = 0.0
    num_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="kd", leave=False)):
        tensors, _ = split_batch(batch, device)

        remaining = num_batches - (step // grad_accum) * grad_accum
        current_accum = min(grad_accum, remaining)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            out = kd_exp.compute_loss(tensors)
            loss = out["loss"] / current_accum

        if not torch.isfinite(out["loss"]):
            raise RuntimeError(f"Non-finite KD loss at batch {step}")

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == num_batches:
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in kd_exp.student.parameters() if p.requires_grad], max_grad_norm
                )
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            step_succeeded = not amp or scaler.get_scale() >= old_scale
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None and step_succeeded:
                scheduler.step()

        tot += out["loss"].item()
        tot_gold += out["gold_loss"].item()
        tot_kd += out["kd_loss"].item()

    n = max(num_batches, 1)
    return tot / n, tot_gold / n, tot_kd / n


def build_truncated_decoder_backbone(n_layers):
    """Smaller student backbone (as in Phase 3): Phase-1 decoder keeping embeddings + first `n_layers` blocks + final_ln."""
    student_config = replace(decoder_config, n_layer=n_layers)
    backbone = DecoderOnlyTransformer(student_config)
    res = backbone.load_state_dict(_load_state(DECODER_CHECKPOINT), strict=False)
    assert not res.missing_keys, res.missing_keys
    assert all(k.startswith("blocks.") and int(k.split(".")[1]) >= n_layers for k in res.unexpected_keys), \
        [k for k in res.unexpected_keys if not k.startswith("blocks.")][:5]
    print(f"Student decoder: {n_layers} layers | dropped tensors from deeper layers: {len(res.unexpected_keys)}")
    return backbone


def sequence_kd_loss(student_out, teacher_out, tensors, T):
    mask = student_out["target_mask"]                       # [B, T] bool, same inputs => same positions for teacher
    s = student_out["logits"][mask].float()                 # [N_tok, V]
    t = teacher_out["logits"][mask].float()
    if s.numel() == 0:
        return student_out["logits"].sum() * 0.0
    kl = F.kl_div(F.log_softmax(s / T, dim=-1), F.softmax(t / T, dim=-1), reduction="batchmean")
    return kl * (T * T)

## 13. Preflight tests (optional, lightweight)

Run when `RUN_PREFLIGHT = True`. No full training; generation is limited to a few tokens. Every check is reported as PASS/FAIL;
a missing checkpoint is reported with its exact path (never skipped), and the summary cell raises if anything failed.

1. Every backbone in `PREFLIGHT_BACKBONES` loads strictly from its checkpoint.
2. Every `encode()` returns `[B, L, D]` and the LM head returns vocabulary logits.
3. Dataset and collator: prompt / target layout, `-100` on prompt and padding, EOS handling.
4. The causal wrapper produces `[B, T, V]` logits, a `target_mask` aligned with the shifted labels, and a finite loss; the encoder-decoder wrapper (current `STANDARD_ENCDEC`) does the same.
5. Right-padding does not change a document's logits on its real positions.
6. `lora` and `lorma` train only adapter parameters (+ task modules) on every backbone.
7. A small batch supports forward, loss and backward with gradients only on trainable parameters.
8. Inference without labels (`loss` is `None`) and a short generation work for both formulations.
9. The KD loss is ~0 for identical teacher/student and positive otherwise.
10. Test predictions cover every document once, in order; empty generations get a fallback; submission files validate.

In [ ]:
PREFLIGHT_RESULTS = []


def preflight_check(name, fn):
    try:
        detail = fn()
        PREFLIGHT_RESULTS.append((name, "PASS", detail or ""))
        print(f"[PASS] {name}" + (f" - {detail}" if detail else ""))
    except Exception as exc:
        PREFLIGHT_RESULTS.append((name, "FAIL", f"{type(exc).__name__}: {exc}"))
        print(f"[FAIL] {name} - {type(exc).__name__}: {exc}")


def _fail(msg):
    raise RuntimeError(msg)


def tiny_docs(n=2):
    """Short copies of validation documents (different lengths) so preflight stays cheap."""
    return [{"id": f"pf_{k}", "document": d["document"][: 400 + 200 * k], "summary": d["summary"][: 150 + 100 * k]}
            for k, d in enumerate(val_docs[:n])]


def tiny_batch(encoder_decoder=False):
    ds = SummDataset(tiny_docs(), tokenizer, max_source_tokens=120, max_target_tokens=48, encoder_decoder=encoder_decoder)
    return summ_collator(ds.items)


def is_adapter_param(name):
    return "lora_" in name or name.endswith(".A") or name.endswith(".B")

In [ ]:
# ---- Tests 1, 2, 6: every backbone -> strict load, encode() + LM logits, every adaptation method ----
pf_backbone = None

if RUN_PREFLIGHT:
    probe_ids = torch.tensor([[BOS_ID] + tokenizer.encode("The appeal is dismissed with costs.") + [EOS_ID]] * 2)
    probe_mask = torch.ones_like(probe_ids)

    for name in PREFLIGHT_BACKBONES:
        info = BACKBONES[name]
        ckpt = info["checkpoint_path"]
        loaded = {}

        def _load():
            if not Path(ckpt).exists():
                raise FileNotFoundError(f"missing checkpoint: {ckpt}")
            bb = load_backbone(name, device="cpu", strict=True)
            loaded["bb"] = bb
            return f"strict load OK ({sum(p.numel() for p in bb.parameters()):,} params)"

        preflight_check(f"1. {name}: checkpoint loads strictly", _load)
        if "bb" not in loaded:
            preflight_check(f"2. {name}: encode() / logits", lambda: _fail("backbone not loaded"))
            continue
        bb = loaded["bb"]

        def _encode():
            bb.eval()
            with torch.no_grad():
                h = bb.encode(probe_ids, probe_mask)
                if name == "encdec":
                    lg = bb(input_ids=probe_ids, attention_mask=probe_mask,
                            decoder_input_ids=torch.full((2, 1), ENCDEC_CONFIG.decoder_start_token_id))["logits"]
                else:
                    lg = bb(input_ids=probe_ids, attention_mask=probe_mask)["logits"]
            assert tuple(h.shape) == (2, probe_ids.shape[1], info["hidden_dim"]), tuple(h.shape)
            assert lg.shape[0] == 2 and lg.shape[-1] == tokenizer.vocab_size, tuple(lg.shape)
            return f"encode {tuple(h.shape)}, logits {tuple(lg.shape)}"

        preflight_check(f"2. {name}: encode() + LM logits", _encode)

        for method in ["lora", "lorma"]:
            def _adapt(method=method):
                cfg = replace(exp_cfg, backbone_name=name, adaptation_method=method)
                m = build_adapted_model(cfg, device="cpu", backbone=copy.deepcopy(bb))
                trainable = [n for n, p in m.named_parameters() if p.requires_grad]
                head = tuple(f"{h}." for h in m.head_module_names())
                wrong = [n for n in trainable if not (is_adapter_param(n) or (head and n.startswith(head)))]
                assert trainable and not wrong, f"unexpected trainable parameters: {wrong[:3]}"
                n_tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
                del m
                return f"trainable params {n_tr:,}"

            preflight_check(f"6. {name}: {method} trainable parameters", _adapt)

        if name == exp_cfg.backbone_name:
            pf_backbone = bb          # reused below; avoids loading the checkpoint again
        else:
            del bb
        loaded.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("RUN_PREFLIGHT=False: preflight skipped.")

In [ ]:
# ---- Tests 3, 4, 5, 7, 8, 9 ----
if RUN_PREFLIGHT:

    def _collator():
        docs = tiny_docs()
        ds = SummDataset(docs, tokenizer, max_source_tokens=120, max_target_tokens=48, encoder_decoder=False)
        b = summ_collator(ds.items)
        for i, it in enumerate(ds.items):
            p, t = it["prompt_ids"], it["target_ids"]
            assert p[0] == BOS_ID and p[-len(tokenizer.encode(SUMMARY_PROMPT)):] == tokenizer.encode(SUMMARY_PROMPT)
            n = len(p) + len(t)
            assert b["input_ids"][i, :n].tolist() == p + t
            assert (b["labels"][i, :len(p)] == -100).all() and (b["labels"][i, n:] == -100).all()
            assert b["labels"][i, len(p):n].tolist() == t
            full = tokenizer.encode(docs[i]["summary"])
            assert t == (full + [EOS_ID] if len(full) + 1 <= 48 else full[:48])
        return f"causal batch {tuple(b['input_ids'].shape)}"

    preflight_check("3. dataset / collator layout", _collator)

    if exp_cfg.backbone_name == "encdec" or pf_backbone is None:
        preflight_check("4-9. model tests", lambda: _fail(
            "these tests use a causal exp_cfg backbone that was loaded in the previous cell"))
    else:
        cb = tiny_batch(encoder_decoder=False)
        ct = {k: v for k, v in cb.items() if torch.is_tensor(v)}

        def _forward():
            m = Phase6SummModel(pf_backbone, BACKBONES[exp_cfg.backbone_name]["hidden_dim"]).eval()
            with torch.no_grad():
                out = m(**ct)
            B, T = ct["input_ids"].shape
            assert tuple(out["logits"].shape) == (B, T, tokenizer.vocab_size)
            assert torch.equal(out["target_mask"][:, :-1], ct["labels"][:, 1:] != -100) and not out["target_mask"][:, -1].any()
            assert torch.isfinite(out["loss"])
            msg = f"causal logits {tuple(out['logits'].shape)}"
            if "encdec" in BACKBONES and Path(BACKBONES["encdec"]["checkpoint_path"]).exists():
                eb = tiny_batch(encoder_decoder=True)
                et = {k: v for k, v in eb.items() if torch.is_tensor(v)}
                em = build_phase6_model("encdec", device="cpu").eval()
                with torch.no_grad():
                    eo = em(**et)
                assert eo["logits"].shape[:2] == et["labels"].shape and torch.isfinite(eo["loss"])
                assert torch.equal(eo["target_mask"], et["labels"] != -100)
                msg += f"; encdec ({'standard' if STANDARD_ENCDEC else 'hierarchical'}) logits {tuple(eo['logits'].shape)}"
                del em
            return msg

        preflight_check("4. wrapper forward (causal + encdec)", _forward)

        def _padding():
            m = Phase6SummModel(pf_backbone, BACKBONES[exp_cfg.backbone_name]["hidden_dim"]).eval()
            n0 = int(ct["attention_mask"][0].sum())
            alone = {k: v[:1, :n0] for k, v in ct.items()}
            with torch.no_grad():
                a = m(**ct)["logits"][0, :n0]
                b = m(**alone)["logits"][0, :n0]
            diff = (a - b).abs().max().item()
            assert diff < 1e-3, f"padding changed logits by {diff:.2e}"
            return f"max diff {diff:.1e}"

        preflight_check("5. right-padding does not change logits", _padding)

        def _backward():
            m = build_adapted_model(exp_cfg, device=DEVICE, backbone=copy.deepcopy(pf_backbone)).train()
            out = m(**{k: v.to(DEVICE) for k, v in ct.items()})
            out["loss"].backward()
            no_grad = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
            leaked = [n for n, p in m.named_parameters() if not p.requires_grad and p.grad is not None]
            assert not no_grad, f"trainable parameters without gradient: {no_grad[:3]}"
            assert not leaked, f"frozen parameters received gradients: {leaked[:3]}"
            n = sum(1 for p in m.parameters() if p.requires_grad)
            del m, out
            return f"{exp_cfg.adaptation_method}: {n} trainable tensors all received gradients"

        preflight_check("7. forward + loss + backward on a small batch", _backward)

        def _inference():
            m = Phase6SummModel(pf_backbone, BACKBONES[exp_cfg.backbone_name]["hidden_dim"]).eval()
            with torch.no_grad():
                out = m(**{k: v for k, v in ct.items() if k != "labels"})
            assert out["loss"] is None and out["target_mask"] is None
            short = replace(exp_cfg, max_generation_length=6, num_beams=1)
            hyps = m.generate_summaries([tiny_docs(1)[0]["document"]], short, tokenizer, device="cpu")
            assert len(hyps) == 1 and isinstance(hyps[0], str)
            msg = f"causal generation ok ({len(hyps[0].split())} words)"
            if Path(BACKBONES["encdec"]["checkpoint_path"]).exists():
                em = build_phase6_model("encdec", device="cpu").eval()
                eh = em.generate_summaries([tiny_docs(1)[0]["document"]], replace(short, backbone_name="encdec"), tokenizer, device="cpu")
                assert len(eh) == 1 and isinstance(eh[0], str)
                msg += f"; encdec generation ok ({len(eh[0].split())} words)"
                del em
            return msg

        preflight_check("8. inference without labels + short generation", _inference)

        def _kd_loss():
            m = Phase6SummModel(pf_backbone, BACKBONES[exp_cfg.backbone_name]["hidden_dim"]).eval()
            with torch.no_grad():
                out = m(**ct)
                same = sequence_kd_loss(out, out, ct, 2.0).item()
                other = {"logits": out["logits"] + torch.randn_like(out["logits"]), "target_mask": out["target_mask"]}
                diff = sequence_kd_loss(other, out, ct, 2.0).item()
            assert abs(same) < 1e-4 and diff > 1e-3, (same, diff)
            return f"identical {same:.1e}, perturbed {diff:.3f}"

        preflight_check("9. KD loss sanity", _kd_loss)

In [ ]:
# ---- Test 10: test predictions + submission files (dummy generator, no backbone) ----
if RUN_PREFLIGHT:

    class _EchoSummarizer(nn.Module):
        """Returns the first words of each document; every third output is empty to exercise the fallback."""
        def generate_summaries(self, documents, cfg, tokenizer, device=DEVICE):
            return ["" if i % 3 == 2 else " ".join(d.split()[:20]) for i, d in enumerate(documents)]

    def _submission():
        preds = run_inference(_EchoSummarizer(), test_docs, replace(exp_cfg, run_name="_preflight"), tokenizer)
        preds = enforce_generation_limit(preds, exp_cfg.max_generation_length)
        validate_submission_format(preds, test_docs, max_summary_tokens=exp_cfg.max_generation_length)
        _, zip_path = save_submission(preds, WORK_DIR / "_preflight_submission")
        verify_zip_root(zip_path)
        return f"{len(preds)} test documents, test-file order, no empty summary"

    preflight_check("10. test predictions and submission validation", _submission)

In [ ]:
# ---- Preflight summary ----
if RUN_PREFLIGHT:
    pf_backbone = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    failed = [r for r in PREFLIGHT_RESULTS if r[1] == "FAIL"]
    print(f"\nPreflight: {len(PREFLIGHT_RESULTS) - len(failed)} passed, {len(failed)} failed")
    for name, status, detail in PREFLIGHT_RESULTS:
        print(f"  {status}  {name}" + (f"  ({detail})" if status == "FAIL" else ""))
    if failed:
        raise RuntimeError("Preflight failed: " + "; ".join(f"{n}: {d}" for n, _, d in failed))

### 13.1 Tiny overfit test (optional)

Train on a handful of pairs; loss should drop sharply and a generated summary should start copying the reference.

In [ ]:
if RUN_TINY_OVERFIT:
    tiny_overfit_test(exp_cfg)
else:
    print("RUN_TINY_OVERFIT=False: skipped.")

## 14. Main training experiment

Trains **only** `exp_cfg` (section 11) when `RUN_TRAINING = True`. Model selection: BERTScore F1 on `val_max_docs` generated
validation summaries (or validation loss when `val_generate = False`). Saves `best_model.pt`, `metrics.json`, `history.json`,
`val_samples.json` and `config.json` in `exp_cfg.run_dir`, and reloads the best checkpoint.

In [ ]:
model = None

if RUN_TRAINING:
    model, best_score, history = fit(exp_cfg, train_loader, val_loader)
else:
    print("RUN_TRAINING=False: training skipped.")

## 15. Knowledge distillation (optional)

`teacher (trained Phase-6 summarizer) → per-token vocabulary distributions (teacher forcing) → smaller student`

KD = `T² · KL(softmax(t/T) ‖ softmax(s/T))` over the summary positions (`target_mask`), total = `(1 − α) · CE(student, gold) + α · KD`.
Teacher and student share the Phase-1 tokenizer, so their vocabularies align.
Token-level KD also needs the **same formulation** for both: a causal teacher (decoder / ssm / cnn) for the causal decoder student
(an `encdec` teacher's logits are over target positions, not prompt+target, so use sequence-level KD for it).
(Sequence-level KD — training the student on teacher-generated summaries — is an alternative: generate with the teacher, then replace the references.)

Runs only when `RUN_KD = True`. Teacher: decoder + LORMA loaded strictly from `KD_TEACHER_CHECKPOINT`. Student: the first
`STUDENT_LAYERS` blocks of the Phase-1 decoder (plus embeddings and final norm) with LORMA.

In [ ]:
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5
KD_LR = 2e-4
KD_EPOCHS = 4
STUDENT_LAYERS = 4

# Teacher: the trained decoder + LORMA run (same settings as the best run, loaded from KD_TEACHER_CHECKPOINT).
teacher_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="lorma",
    learning_rate=5e-4,
    batch_size=BATCH_SIZE,
    grad_accum=16,
    epochs=4,
    lorma_rank=16,
    lorma_alpha=32.0,
    max_generation_length=384,
    num_beams=1,
    length_penalty=1.0,
    val_generate=True,
    val_max_docs=50,
    run_name="dec_lorma",
)

# Smaller student
student_cfg = replace(
    teacher_cfg,
    learning_rate=KD_LR,
    epochs=KD_EPOCHS,
    run_name="decoder_lorma_kd_student",
)


def build_kd_teacher(checkpoint_path=None):
    path = Path(checkpoint_path or KD_TEACHER_CHECKPOINT)
    if not path.is_file():
        raise FileNotFoundError(
            f"KD teacher checkpoint not found: {path}. Set KD_TEACHER_CHECKPOINT in section 2."
        )
    teacher = build_adapted_model(teacher_cfg)
    teacher.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    print("Teacher loaded:", path)
    return teacher


def build_kd_student():
    # Student: smaller decoder initialised from Phase-1 pretrained weights
    return build_adapted_model(student_cfg, backbone=build_truncated_decoder_backbone(STUDENT_LAYERS))


def build_kd_pair():
    teacher = build_kd_teacher()     # fails early if the teacher checkpoint is missing
    student = build_kd_student()
    return teacher, student


def check_kd_pair(teacher, student):
    """Teacher and student see the same batch and produce aligned logits / target masks."""
    batch = next(iter(train_loader))
    tensors, _ = split_batch(batch, DEVICE)
    with torch.no_grad():
        t_out = teacher(**tensors)
        s_out = student.eval()(**tensors)
    student.train()
    assert t_out["logits"].shape == s_out["logits"].shape
    assert torch.equal(t_out["target_mask"], s_out["target_mask"])
    print("KD compatibility checks passed. teacher loss", round(t_out["loss"].item(), 4),
          "| student loss", round(s_out["loss"].item(), 4))


def run_kd():
    teacher, student = build_kd_pair()
    check_kd_pair(teacher, student)
    # teacher keeps labels so both models see identical inputs; KD uses the student's target_mask
    kd_exp = DistillationExperiment(teacher, student, sequence_kd_loss,
                                    temperature=KD_TEMPERATURE, alpha=KD_ALPHA)
    optimizer = build_optimizer(student, student_cfg)
    scheduler = build_scheduler(optimizer, student_cfg, len(train_loader))

    run_dir = student_cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(student_cfg)
    best_score = -float("inf")

    for epoch in range(student_cfg.epochs):
        loss, gold, kd = train_kd_epoch(kd_exp, train_loader, optimizer, DEVICE,
                                        grad_accum=student_cfg.grad_accum, use_amp=student_cfg.use_amp,
                                        scheduler=scheduler)
        val_out = run_validation(student, val_loader, val_docs, student_cfg, generate=student_cfg.val_generate)
        if student_cfg.val_generate:
            metrics = compute_summ_metrics(val_out["hypotheses"], val_out["references"])
            score = metrics["bertscore_f1"]
        else:
            metrics, score = {}, -val_out["loss"]
        print(f"epoch {epoch+1}: loss={loss:.4f} gold={gold:.4f} kd={kd:.4f} val_loss={val_out['loss']:.4f} {metrics}")

        if score > best_score:
            best_score = score
            torch.save(student.state_dict(), run_dir / "best_model.pt")
            with open(run_dir / "metrics.json", "w") as f:
                json.dump({**metrics, "val_loss": val_out["loss"], "temperature": KD_TEMPERATURE,
                           "alpha": KD_ALPHA, "student_layers": STUDENT_LAYERS,
                           "teacher": teacher_cfg.run_name}, f, indent=2)
    student.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=DEVICE, weights_only=True))
    return student, best_score


def load_kd_student(checkpoint_path=None):
    path = Path(checkpoint_path or (student_cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"KD student checkpoint not found: {path}")
    student = build_kd_student()
    student.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
    student.eval()
    print("Loaded KD student:", path)
    return student


def kd_efficiency_report(student):
    teacher = build_kd_teacher()
    t_total = sum(p.numel() for p in teacher.parameters())
    s_total = sum(p.numel() for p in student.parameters())
    s_train = sum(p.numel() for p in student.parameters() if p.requires_grad)
    del teacher
    gc.collect()
    report = {"teacher_total_params": t_total, "student_total_params": s_total,
              "student_trainable_params": s_train, "param_reduction_pct": round(100 * (1 - s_total / t_total), 2)}
    print(json.dumps(report, indent=2))
    return report

### 15.1 Run knowledge distillation

In [ ]:
kd_student = None

if RUN_KD:
    assert not ENCODER_DECODER, "KD uses the causal formulation: set ENCODER_DECODER = False"
    kd_student, kd_best = run_kd()
    print("Best KD score:", kd_best)
    kd_metadata = {"teacher": teacher_cfg.run_name, "teacher_checkpoint": str(KD_TEACHER_CHECKPOINT),
                   "student_layers": STUDENT_LAYERS, "temperature": KD_TEMPERATURE, "alpha": KD_ALPHA,
                   "best_score": kd_best, **kd_efficiency_report(kd_student)}
    with open(student_cfg.run_dir / "kd_metadata.json", "w", encoding="utf-8") as f:
        json.dump(kd_metadata, f, indent=2)
else:
    print("RUN_KD=False: knowledge distillation skipped.")

## 16. Test inference (generation)

`INFERENCE_SOURCE = "main"` uses the `exp_cfg` model (trained above, or loaded from `INFERENCE_CHECKPOINT` /
`exp_cfg.run_dir/best_model.pt`); `"kd"` uses the KD student. Summaries are generated with the configuration's decoding
settings, empty generations get a first-sentence fallback, and every summary is trimmed to `max_generation_length` tokens.
Each source writes to its own run directory.

In [ ]:
if RUN_INFERENCE:
    if INFERENCE_SOURCE == "main":
        inference_cfg = exp_cfg
        inference_model = model if model is not None else load_trained_model(exp_cfg, INFERENCE_CHECKPOINT)
    elif INFERENCE_SOURCE == "kd":
        inference_cfg = student_cfg
        inference_model = kd_student if kd_student is not None else load_kd_student(INFERENCE_CHECKPOINT)
    else:
        raise ValueError(f"INFERENCE_SOURCE must be 'main' or 'kd', got {INFERENCE_SOURCE!r}")

    inference_model = inference_model.to(DEVICE).eval()
    print("Inference source:", INFERENCE_SOURCE, "| run:", inference_cfg.run_dir.name)

    test_predictions = run_inference(inference_model, test_docs, inference_cfg, tokenizer, save_every=10)
    test_predictions = enforce_generation_limit(test_predictions, inference_cfg.max_generation_length)
    print("Mean summary words:", np.mean([len(p["summary"].split()) for p in test_predictions]))
else:
    print("RUN_INFERENCE=False: test inference skipped.")

## 17. Submission validation and ZIP

Expected `prediction.json` — a JSON **list**, exactly one generated summary per test document:

```json
[
  {"id": "doc_001", "summary": "Generated text..."},
  {"id": "doc_002", "summary": "Generated text..."}
]
```

Generate from a **freshly loaded** best checkpoint with the decoding settings recorded in `exp_cfg`.

In [ ]:
if RUN_INFERENCE:
    validate_submission_format(test_predictions, test_docs, max_summary_tokens=inference_cfg.max_generation_length)

    pred_path, zip_path = save_submission(test_predictions, inference_cfg.run_dir)
    verify_zip_root(zip_path)
    print("Final submission ready:", zip_path)
else:
    print("RUN_INFERENCE=False: no submission written.")

## Recorded Codabench results (Phase 6)

| Run | Backbone | Adaptation | Formulation | Codabench BERTScore F1 | Test summaries |
|---|---|---|---|---|---|
| dec_lorma | decoder | LORMA | causal prompt + summary | **0.7876** (leaderboard submission) | mean 338 words |
| KD | decoder, 4-layer student | KD (teacher: decoder + LORMA) + LORMA | causal | 0.7542 | mean 368 words |
| encderlora | encdec | LoRA | standard encoder-decoder | 0.7309 | every summary 384 words |
| encdeclorma | encdec | LORMA | standard encoder-decoder | 0.7152 | every summary 384 words |
